# CS-781 Phase 4 CJPE — Robust Full-Epoch Rebuild

This notebook starts from the actual Phase-4 train/dev/test data and the existing Phase-1 **Architecture A** checkpoint. It does **not** introduce a new backbone architecture. The aim is to repair the most important weakness in the prior run: random mini-batches sampled from the full dataset without completing epochs.

## Main changes
- Uses **all official training records** in shuffled epochs, without replacement within each epoch.
- Evaluates on the entire development split (1,988 records) and saves the best dev checkpoint.
- Uses a lower adapter learning rate, cosine schedule, gradient clipping, and masked-mean + masked-max + last-valid-token pooling.
- Tokenization is cached in chunks and resumes after an interrupted tokenization cell.
- Training checkpoints save the trainable adapter delta, classifier head, optimizer, scheduler, AMP scaler, RNG, epoch permutation, and cursor; the frozen ~1 GB backbone is not rewritten every save.
- Writes a portable `CJPE_PHASE4_RESUME_BUNDLE.zip` and a schema-validated `submission.zip`.

## Run instructions
Attach the same official Phase-4 dataset, the existing Phase-1 Architecture A checkpoint, and the matching LEGAL_40K tokenizer. Select GPU. Run All. If the training cell reaches its 7-hour session budget, rerun **Cell 7** in the same working directory to continue. For a fresh session, attach the downloaded `CJPE_PHASE4_RESUME_BUNDLE.zip` as an input and Run All.

**Important limits:** this notebook focuses on a reliable full-data Architecture A run first. It does not claim that the current run satisfies the separate multi-architecture experiment/KD rubric, because B/C checkpoint compatibility and the D checkpoint need validation. No hidden-test score, including 0.90+, can be guaranteed. Use the full development results to decide whether the next experiment is worth the compute.


In [ ]:
import os, sys, json, re, gc, math, time, random, hashlib, zipfile, shutil, warnings, ast
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report
from tokenizers import Tokenizer

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
SEED = 781
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)} | VRAM={torch.cuda.get_device_properties(i).total_memory/1e9:.2f} GB")
else:
    print("WARNING: no GPU detected. This four-hour workflow is not practical on CPU.")

ROOT = Path("/kaggle/working/CS781_PHASE4_CJPE")
DATA_DIR = ROOT / "data"
EXP_DIR = ROOT / "experiments"
SUB_DIR = ROOT / "submissions"
for d in [ROOT, DATA_DIR, EXP_DIR, SUB_DIR]: d.mkdir(parents=True, exist_ok=True)
TOKENIZER_SHA256 = "3e38dd5c561d84ad8fbadf792bbca490f8c70346d922df7afbef37c5a942b71e"
print("Output root:", ROOT)

In [ ]:
# This cell will fail with an actionable error if the official data/checkpoints are not attached or accessible.

OFFICIAL_REPO_ID = "Exploration-Lab/CS-781-Capstone"

def read_hf_secret():
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
    if token: return token
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for name in ("HF", "HF_TOKEN"):
            try:
                v = client.get_secret(name)
                if v and v.strip(): return v.strip()
            except Exception:
                pass
    except Exception:
        pass
    return None

HF_TOKEN = read_hf_secret()

# ---------- Attach-and-run input recovery ----------
# Kaggle mounts attached datasets under /kaggle/input. Selectively extract only
# relevant members from attached ZIPs, avoiding unnecessary legal-corpus archives.
AUTO_EXTRACT_DIR = ROOT / "auto_extracted"
AUTO_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
WEIGHT_EXTS = {".pt", ".pth", ".ckpt"}
RELEVANT_WEIGHT_NAMES = {"best_model.pt", "latest.pt", "model.pt", "checkpoint.pt", "best.pt", "model.pth", "checkpoint.pth"}

# Fast bounded traversal: Phase-1 recovery datasets contain huge token_cache_40k trees with many .bin shards.
# These shards are not model checkpoints and must never be inventoried one by one.
_SKIP_DIR_NAMES = {
    "token_cache_40k", "token_cache", "corpus_cache", "hf_cache", ".cache",
    "__pycache__", ".git", "node_modules", "lost+found"
}
def iter_files_pruned(base, suffixes=None, names=None):
    base = Path(base)
    if not base.exists():
        return
    wanted_suffixes = {x.lower() for x in suffixes} if suffixes else None
    wanted_names = {x.lower() for x in names} if names else None
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d.lower() not in _SKIP_DIR_NAMES]
        for filename in files:
            p = Path(root) / filename
            low = filename.lower()
            if wanted_names is not None and low not in wanted_names:
                continue
            if wanted_suffixes is not None and p.suffix.lower() not in wanted_suffixes:
                continue
            yield p

def _safe_zip_target(root, member):
    member = member.replace("\\", "/")
    if member.startswith("/") or re.match(r"^[A-Za-z]:", member):
        return None
    target = (root / member).resolve()
    try:
        target.relative_to(root.resolve())
    except Exception:
        return None
    return target

def selectively_extract_attached_zips():
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        print("No /kaggle/input found; continuing with /kaggle/working files.")
        return
    archives = sorted(iter_files_pruned(input_root, suffixes={".zip"}))
    if not archives:
        print("No ZIP inputs found; recursive direct-file discovery will be used.")
        return
    total = 0
    for archive in archives:
        try:
            digest = hashlib.sha1(str(archive).encode("utf-8")).hexdigest()[:10]
            dest_root = AUTO_EXTRACT_DIR / (archive.stem[:50] + "_" + digest)
            with zipfile.ZipFile(archive) as zf:
                selected = []
                for info in zf.infolist():
                    if info.is_dir():
                        continue
                    name = info.filename.replace("\\", "/")
                    base = Path(name).name.lower()
                    suffix = Path(base).suffix.lower()
                    data_file = base in {"train.json", "dev.json", "test.json"}
                    tokenizer_file = base in {"tokenizer.json", "legal_40k_tokenizer.json"}
                    weight_file = suffix in WEIGHT_EXTS and (
                        base in RELEVANT_WEIGHT_NAMES or
                        any(k in name.lower() for k in ("checkpoint", "phase1", "phase_1", "architecture_a", "architecture_b", "architecture_c", "architecture_d", "arch-a", "arch-b", "arch-c", "arch-d", "a_large", "legalhybrid", "dmlc", "prefixformer", "model/", "models/"))
                    )
                    if data_file or tokenizer_file or weight_file:
                        target = _safe_zip_target(dest_root, info.filename)
                        if target is not None and info.file_size <= 6 * 1024**3:
                            selected.append((info, target))
                size_sum = sum(i.file_size for i, _ in selected)
                if size_sum > 14 * 1024**3:
                    print("Skip ZIP extraction; relevant members exceed 14 GiB:", archive)
                    continue
                written = 0
                for info, target in selected:
                    if target.exists() and target.stat().st_size == info.file_size:
                        continue
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with zf.open(info, "r") as src, open(target, "wb") as dst:
                        shutil.copyfileobj(src, dst, length=1024*1024)
                    written += 1
                total += written
                if written:
                    print(f"ZIP recovery: {archive.name} -> {written} relevant file(s) in {dest_root}")
        except Exception as e:
            print("ZIP recovery warning:", archive, type(e).__name__, str(e)[:180])
    print("ZIP recovery complete; relevant files extracted:", total)

print("[CJPE AUTO-DISCOVERY FINAL v2] Using expanded schema parser; duplicate train/dev IDs are allowed.")
print("[1/5] Recovering relevant files from attached ZIPs...")
selectively_extract_attached_zips()

def write_input_inventory():
    # Inventory only files that can matter for this task; do not enumerate token-cache shards.
    wanted_names={"train.json","dev.json","test.json","tokenizer.json","legal_40k_tokenizer.json",
                  "best_model.pt","latest.pt","model.pt","checkpoint.pt","best.pt",
                  "model.pth","checkpoint.pth"}
    wanted_suffixes={".json",".pt",".pth",".ckpt",".zip"}
    rows=[]
    for base in (Path("/kaggle/input"), AUTO_EXTRACT_DIR):
        for p in iter_files_pruned(base, suffixes=wanted_suffixes):
            low=p.name.lower()
            pathlow=str(p).lower()
            if low not in wanted_names and not (p.suffix.lower() in {".pt",".pth",".ckpt"} and any(k in low for k in ("checkpoint","best_model","latest","model"))):
                # Retain JSON only when it is one of the known dataset/tokenizer filenames.
                if p.suffix.lower()==".json" and low not in wanted_names: continue
                if p.suffix.lower()==".zip": pass
                elif p.suffix.lower() not in {".pt",".pth",".ckpt"}: continue
            try: size=p.stat().st_size
            except Exception: size=-1
            rows.append({"path":str(p),"suffix":p.suffix.lower(),"size_bytes":size})
    if rows:
        inventory=pd.DataFrame(rows).drop_duplicates(subset=["path"]).sort_values(["suffix","path"])
        inventory.to_csv(ROOT/"input_inventory.csv",index=False)
        print("Relevant input inventory saved:", ROOT/"input_inventory.csv", "| files:", len(inventory))
        print(inventory.head(100).to_string(index=False))
print("[2/5] Building targeted input inventory (token-cache shards excluded)...")
write_input_inventory()

def parse_label(value):
    if isinstance(value, (int, np.integer)) and int(value) in (0,1): return int(value)
    if isinstance(value, (bool, np.bool_)): return int(value)
    if isinstance(value, float) and value in (0.0,1.0): return int(value)
    s = str(value).strip().lower()
    if s in {"0", "0.0", "false", "reject", "rejected", "dismissed", "denied", "refused", "negative", "not allowed"}: return 0
    if s in {"1", "1.0", "true", "accept", "accepted", "allowed", "allow", "granted", "positive", "success"}: return 1
    raise ValueError(f"Unsupported binary label value: {value!r}")

def normalize_records(raw, labeled):
    """Normalize common CJPE JSON layouts. Training rows may legitimately repeat case IDs."""
    # Unwrap common container objects even when they also contain metadata.
    wrapper_keys=("data","records","examples","items","train","dev","test","dataset","instances","annotations")
    for _ in range(8):
        if isinstance(raw, dict):
            picked=next((k for k in wrapper_keys if k in raw and isinstance(raw[k], (list,dict))), None)
            if picked is not None:
                raw=raw[picked]
                continue
        break
    records=[]
    id_keys=("id","case_id","caseId","caseID","document_id","example_id","uid","case_no","case_number")
    text_keys=("text","case_text","document","content","judgment_text","judgement_text","case_document","full_text","judgment","judgement","case")
    label_keys=("label","labels","outcome","target","y","decision","pred","label_id","class","class_label","judgment_label","judgement_label","outcome_label","decision_label","final_outcome","result","disposition","judgment_outcome","judgement_outcome")
    if isinstance(raw, dict):
        # Some files are keyed by case ID; preserve that key as the default ID.
        items=[]
        for k,v in raw.items():
            if isinstance(v, dict):
                obj=dict(v); obj.setdefault("id",k); items.append(obj)
            else:
                return None
    elif isinstance(raw, list):
        items=raw
    else:
        return None
    for i,obj in enumerate(items):
        if not isinstance(obj, dict): return None
        # Match case-insensitively while preserving source field priority.
        lower={str(k).lower():k for k in obj}
        rid_key=next((lower[k.lower()] for k in id_keys if k.lower() in lower), None)
        txt_key=next((lower[k.lower()] for k in text_keys if k.lower() in lower and isinstance(obj[lower[k.lower()]],str)), None)
        lab_key=next((lower[k.lower()] for k in label_keys if k.lower() in lower), None) if labeled else None
        rid=obj[rid_key] if rid_key is not None else None
        txt=obj[txt_key] if txt_key is not None else None
        lab=obj[lab_key] if lab_key is not None else None
        # Support labels nested as a one-key object, e.g. {"label": {"value": 1}}.
        if labeled and isinstance(lab, dict) and len(lab)==1:
            lab=next(iter(lab.values()))
        if rid is None or not isinstance(txt,str) or not txt.strip(): return None
        row={"id":str(rid), "id_value":rid, "text":txt}
        if labeled:
            if lab is None: return None
            try: row["label"]=parse_label(lab)
            except Exception: return None
        records.append(row)
    if not records: return None
    # Do not reject train/dev just because the official preprocessing contains repeated IDs.
    # Test IDs are checked for uniqueness in load_records below.
    return records

def phase4_path_score(p, kind):
    s=str(p).lower().replace("\\","/")
    score=0
    if p.name.lower()==f"{kind}.json": score += 50
    if "phase_4_cjpe" in s or "phase-4-cjpe" in s or "phase4_cjpe" in s: score += 100
    if "cjpe" in s or "court_judgment" in s: score += 70
    if "phase_4" in s or "phase4" in s: score += 60
    if "/train/" in s and kind=="train": score += 10
    if "/test/" in s and kind=="test": score += 10
    if "/dev/" in s and kind=="dev": score += 10
    if "phase_1" in s or "phase1" in s: score -= 100
    return score

def discover_local_phase4():
    bases=[Path("/kaggle/input"), Path("/kaggle/working")]
    all_json=[]
    for base in bases:
        if base.exists():
            try:
                all_json.extend(p for p in iter_files_pruned(base, suffixes={".json"}, names={"train.json","dev.json","test.json"}) if p.is_file() and p.stat().st_size < 2_500_000_000)
            except Exception as e:
                print("JSON scan warning:", base, e)
    all_json=list(set(all_json))
    exact={k:[p for p in all_json if p.name.lower()==f"{k}.json"] for k in ("train","dev","test")}

    def valid_path(p, kind):
        try:
            with open(p,"r",encoding="utf-8") as f: raw=json.load(f)
            recs=normalize_records(raw,labeled=(kind!="test"))
            return bool(recs) and (kind=="test" or all("label" in x for x in recs))
        except Exception:
            return False

    # First score exact filenames by recognizable Phase-4 paths.
    found={}
    for kind in ("train","dev","test"):
        candidates=[p for p in exact[kind] if phase4_path_score(p,kind)>0]
        candidates.sort(key=lambda p:(phase4_path_score(p,kind),p.stat().st_size),reverse=True)
        for p in candidates:
            if valid_path(p,kind):
                found[kind]=p; break

    # If the Kaggle dataset has generic folder names, identify a sibling train/dev/test triplet by schema.
    # Enumerate only ancestors near files named exactly train.json/dev.json/test.json to keep this scan bounded.
    if len(found)<3:
        roots=set()
        for p in all_json:
            if p.name.lower() not in {"train.json","dev.json","test.json"}: continue
            a=p.parent
            for _ in range(4):
                roots.add(a)
                if a.parent==a: break
                a=a.parent
        triplets=[]
        for root in roots:
            chosen={}
            for kind in ("train","dev","test"):
                inside=[p for p in exact[kind] if root==p.parent or root in p.parents]
                inside.sort(key=lambda p:(phase4_path_score(p,kind),-len(p.relative_to(root).parts),p.stat().st_size),reverse=True)
                for p in inside:
                    if valid_path(p,kind):
                        chosen[kind]=p; break
            if len(chosen)==3:
                score=sum(phase4_path_score(chosen[k],k) for k in chosen) - 2*sum(len(chosen[k].relative_to(root).parts) for k in chosen)
                triplets.append((score,root,chosen))
        if triplets:
            triplets.sort(key=lambda x:(x[0],str(x[1])),reverse=True)
            score,root,chosen=triplets[0]
            # Prefer a full schema-valid triplet over independently guessed files.
            found=chosen
            print("Schema-matched Phase-4 file triplet root:",root,"| score:",score)
    return found

print("[3/5] Discovering local Phase-4 files...")
DATA_PATHS=discover_local_phase4()
print("Local Phase-4 candidate paths:", {k:str(v) for k,v in DATA_PATHS.items()})

def official_hf_fallback(missing_kinds):
    if not missing_kinds:
        return {}
    if not HF_TOKEN:
        print("No Hugging Face token found in Kaggle Secret HF/HF_TOKEN; trying public/gated metadata without a token.")
    try:
        from huggingface_hub import HfApi, hf_hub_download
        api=HfApi(token=HF_TOKEN)
        files=api.list_repo_files(repo_id=OFFICIAL_REPO_ID, repo_type="dataset")
        lower=[(f,f.lower()) for f in files]
        resolved={}
        for kind in missing_kinds:
            possibles=[]
            for f,lo in lower:
                if not lo.endswith(f"/{kind}.json") and lo != f"{kind}.json": continue
                phase_match=("phase_4_cjpe" in lo or "phase-4-cjpe" in lo or "phase4_cjpe" in lo or "cjpe" in lo or "phase_4" in lo)
                if phase_match: possibles.append(f)
            if not possibles:
                for f,lo in lower:
                    if lo.endswith(f"/{kind}.json") or lo==f"{kind}.json": possibles.append(f)
            if not possibles:
                print("Official HF repo had no matching",kind,"JSON file")
                continue
            possibles.sort(key=lambda f:("phase_4_cjpe" in f.lower() or "cjpe" in f.lower(), "train" in f.lower() if kind=="train" else True), reverse=True)
            dest=hf_hub_download(repo_id=OFFICIAL_REPO_ID, filename=possibles[0], repo_type="dataset", token=HF_TOKEN)
            resolved[kind]=Path(dest)
            print("Downloaded official",kind,"from repo path:",possibles[0])
        return resolved
    except Exception as e:
        print("Official Hugging Face fallback failed:",type(e).__name__,str(e)[:500])
        return {}

DATA_PATHS.update(official_hf_fallback([k for k in ("train","dev","test") if k not in DATA_PATHS]))
missing=[k for k in ("train","dev","test") if k not in DATA_PATHS]
if missing:
    raise RuntimeError(
        "Missing official Phase-4 files: " + str(missing) + ". Attach the Kaggle dataset containing Phase_4_CJPE/Train/train.json, Train/dev.json, Test/test.json, or enable Internet and add a working Kaggle Secret named HF (or HF_TOKEN) with access to Exploration-Lab/CS-781-Capstone. Do not substitute outside data."
    )

def load_records(kind):
    p=DATA_PATHS[kind]
    with open(p,"r",encoding="utf-8") as f: raw=json.load(f)
    recs=normalize_records(raw,labeled=(kind!="test"))
    if not recs:
        if isinstance(raw, dict):
            print(f"{kind} schema diagnostic | top-level type=dict | root keys={list(raw.keys())[:30]}")
            probe=next((v for v in raw.values() if isinstance(v,dict)),None)
            if isinstance(probe,dict): print(f"{kind} schema diagnostic | first record keys={list(probe.keys())[:50]}")
        elif isinstance(raw, list) and raw and isinstance(raw[0],dict):
            print(f"{kind} schema diagnostic | top-level type=list | first record keys={list(raw[0].keys())[:50]}")
            print(f"{kind} schema diagnostic | first record sample={str(raw[0])[:1200]}")
        raise RuntimeError(f"Could not normalize Phase-4 {kind} schema at {p}. See the printed schema diagnostics above.")
    ids=[x["id"] for x in recs]
    duplicate_count=len(ids)-len(set(ids))
    if kind=="test" and duplicate_count:
        raise RuntimeError(f"Official test split has {duplicate_count} repeated IDs; refusing to write an invalid submission. Check that the official Phase_4_CJPE/Test/test.json was selected.")
    if kind in ("train","dev") and duplicate_count:
        print(f"{kind.upper()} NOTE: retaining {duplicate_count:,} repeated-ID rows from the combined single-/multi-instance split; they are not automatically duplicates to remove.")
    print(f"{kind:5s}: {len(recs):,} records | unique_ids={len(set(ids)):,} | labels={dict(Counter(x['label'] for x in recs)) if kind!='test' else 'hidden'} | {p}")
    return recs
TRAIN_RECORDS=load_records("train")
DEV_RECORDS=load_records("dev")
TEST_RECORDS=load_records("test")
assert set(x["label"] for x in TRAIN_RECORDS)=={0,1}, "Training split must contain both binary classes"
assert set(x["label"] for x in DEV_RECORDS)=={0,1}, "Development split must contain both binary classes"
train_dev_overlap=set(x['id'] for x in TRAIN_RECORDS)&set(x['id'] for x in DEV_RECORDS)
print("Train/dev ID overlap:",len(train_dev_overlap))
if train_dev_overlap:
    print("WARNING: removing train examples whose IDs are also in dev to prevent leakage.")
    TRAIN_RECORDS=[x for x in TRAIN_RECORDS if x['id'] not in train_dev_overlap]
    assert set(x['label'] for x in TRAIN_RECORDS)=={0,1}, "Both classes must remain after removing train/dev ID overlaps"
print("Majority-class baseline Macro-F1 (dev):",round(f1_score([x['label'] for x in DEV_RECORDS],np.full(len(DEV_RECORDS),Counter(x['label'] for x in TRAIN_RECORDS).most_common(1)[0][0]),average="macro",zero_division=0),4))

# Checkpoint candidates: model weights are always selected from your own Phase-1 artifacts.
ARCH_CLASS_NAMES={"A":"LegalDecoderA","B":"LegalHybrid150","C":"DMLC150","D":"LegalPrefixFormer145"}
PARAM_TARGETS={"A":243_897_344,"B":149_902_080,"C":149_061_510,"D":144_786_688}

def checkpoint_state(payload):
    for k in ("model_state_dict","state_dict","model_weights","weights","model"):
        v=payload.get(k) if isinstance(payload,dict) else None
        if isinstance(v,dict) and v and any(torch.is_tensor(z) for z in v.values()):
            return v
    if isinstance(payload,dict) and payload and all(torch.is_tensor(v) for v in payload.values()):
        return payload
    return None

def payload_config(payload):
    for k in ("model_config","MODEL_CONFIG","config","cfg","model_cfg","model_config_dict"):
        v=payload.get(k) if isinstance(payload,dict) else None
        if isinstance(v,dict) and ("vocab_size" in v or "d_model" in v): return v
    return {}

def infer_arch_from_payload(payload,path,state=None):
    cfg=payload_config(payload)
    names=[]
    for key in ("architecture","architecture_name","model_name","name"):
        if isinstance(cfg.get(key),str): names.append(cfg[key].lower())
        if isinstance(payload.get(key),str): names.append(payload[key].lower())
    plan=payload.get("training_plan",{})
    if isinstance(plan,dict):
        for key in ("architecture","architecture_name","model_name"):
            if isinstance(plan.get(key),str): names.append(plan[key].lower())
    s=" ".join(names)+" "+str(path).lower()
    if "prefix" in s or "legalprefixformer" in s or "prefix_length" in cfg: return "D",cfg
    if "dmlc" in s or "case_memory_slots" in cfg or "encoder_layers" in cfg: return "C",cfg
    if "legalhybrid" in s or "arch-b" in s or "architecture-b" in s or "master_b" in s or "archb" in s: return "B",cfg
    if "legaldecoder" in s or "a_large" in s or "arch-a" in s or "architecture-a" in s or "archa" in s: return "A",cfg
    d=int(cfg.get("d_model",0) or 0); n=int(cfg.get("n_layers",0) or 0)
    if d>=1000 or d==1024: return "A",cfg
    if "case_memory_slots" in cfg or d==640: return "C",cfg
    if d==768 and n>=22: return "B",cfg
    if d==768 and ("prefix_length" in cfg or n==20): return "D",cfg
    return None,cfg

print("[4/5] Discovering canonical checkpoints (excluding token-cache directories)...")
candidate_paths=[]
for base in (Path("/kaggle/input"), Path("/kaggle/working")):
    if base.exists():
        try:
            for p in iter_files_pruned(base, suffixes={".pt", ".pth", ".ckpt"}):
                # Most periodic checkpoints add no value and are costly to deserialize; prefer canonical saved weights.
                low_name=p.name.lower()
                canonical=(low_name in RELEVANT_WEIGHT_NAMES or "checkpoint" in low_name or "best_model" in low_name)
                if not canonical: continue
                low_path=str(p).lower().replace("\\", "/")
                if ROOT == p.parent or ROOT in p.parents:
                    if AUTO_EXTRACT_DIR != p.parent and AUTO_EXTRACT_DIR not in p.parents:
                        continue
                if EXP_DIR == p.parent or EXP_DIR in p.parents or SUB_DIR == p.parent or SUB_DIR in p.parents:
                    continue
                if any(marker in low_path for marker in ("phase4_cjpe", "phase_4_cjpe", "phase-4-cjpe", "/phase4/", "/phase_4/", "/cjpe/")):
                    continue
                candidate_paths.append(p)
        except Exception as e:
            print("Checkpoint scan warning:",e)
# Do not load dozens of duplicate checkpoints from multiple recovery snapshots. Preserve one/few candidates per artifact path.
def _cp_priority(p):
    nm=p.name.lower(); lo=str(p).lower()
    name_rank={"best_model.pt":0,"latest.pt":1,"model.pt":2,"checkpoint.pt":3,"best.pt":4,"model.pth":5,"checkpoint.pth":6}.get(nm,7)
    return (name_rank,len(str(p)),str(p))
def _path_arch(p):
    lo=str(p).lower().replace("_", "-").replace(" ", "-")
    if any(k in lo for k in ("architecture-a","arch-a","arch-a-","a-large","master-a","legaldecodera")): return "A"
    if any(k in lo for k in ("architecture-b","arch-b","legalhybrid","master-b")): return "B"
    if any(k in lo for k in ("architecture-c","arch-c","c-arch","dmlc","master-c")): return "C"
    if any(k in lo for k in ("architecture-d","arch-d","prefixformer","master-d")): return "D"
    return None
# Balance checkpoint probing across architecture-named datasets. Unknown-path candidates are a small fallback set.
_candidate_sorted=sorted(set(candidate_paths),key=_cp_priority)
_balanced={a:[] for a in "ABCD"}; _unknown=[]
for _p in _candidate_sorted:
    _a=_path_arch(_p)
    if _a and len(_balanced[_a])<2: _balanced[_a].append(_p)
    elif _a is None and len(_unknown)<3: _unknown.append(_p)
candidate_paths=[]
for _a in "ABCD": candidate_paths.extend(_balanced[_a])
candidate_paths.extend(_unknown)
print("Canonical checkpoint candidates to inspect (balanced and capped):",len(candidate_paths),"| per arch:",{a:len(v) for a,v in _balanced.items()},"| unknown:",len(_unknown))
# This first robust run uses the known-compatible Architecture A checkpoint. Other architectures are left for a later validated comparison; this avoids loading untested checkpoints before the main full-epoch run.
meta=[]
_a_candidates=list(_balanced.get("A", []))
if not _a_candidates:
    _a_candidates=[p for p in _candidate_sorted if _path_arch(p)=="A"][:2]
for j,p in enumerate(_a_candidates[:2]):
    print(f"Inspecting Architecture-A checkpoint {j+1}/{min(2,len(_a_candidates))}: {p}", flush=True)
    try:
        try:
            payload=torch.load(p,map_location="cpu",weights_only=False,mmap=True)
        except (TypeError, RuntimeError, ValueError):
            payload=torch.load(p,map_location="cpu",weights_only=False)
        st=checkpoint_state(payload)
        if st is None:
            del payload
            continue
        arch,cfg=infer_arch_from_payload(payload,p,st)
        if arch != "A":
            del payload,st
            continue
        if not cfg or not ("d_model" in cfg and "vocab_size" in cfg):
            print("Skip checkpoint without embedded compatible model_config:",p,flush=True)
            del payload,st
            continue
        vals=[]
        for k in ("validation_loss","best_val_loss","best_validation_loss","val_loss"):
            v=payload.get(k)
            if isinstance(v,(int,float)) and np.isfinite(v): vals.append(float(v))
        step=int(payload.get("step",payload.get("global_step",0)) or 0)
        meta.append({"arch":"A","path":str(p),"val_loss":min(vals) if vals else float("inf"),"step":step,"cfg":cfg,"state_keys":len(st),"basename":p.name,"suffix":p.suffix.lower()})
        del payload,st
        break
    except Exception as e:
        print("Could not inspect this Architecture-A checkpoint:",type(e).__name__,str(e)[:300],flush=True)
        continue

CHECKPOINT_PATHS={}
CHECKPOINT_META={}
if meta:
    pick=meta[0]
    CHECKPOINT_PATHS["A"]=Path(pick["path"])
    CHECKPOINT_META["A"]=pick
    print(f"CHECKPOINT A: {pick['path']} | step={pick['step']} | val_loss={pick['val_loss']}",flush=True)
else:
    raise RuntimeError("Could not load a compatible Architecture-A checkpoint from the attached inputs. Check the A dataset attachment.")
AVAILABLE_ARCHS=["A"]
missing_arch=["B","C","D"]  # this direct-run notebook deliberately executes the focused A experiment only
print("Focused run architectures:",AVAILABLE_ARCHS,"| other architecture checkpoints not inspected: B, C, D",flush=True)

# Use the exact course LEGAL_40K tokenizer fingerprint when available.
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()

print("[5/5] Locating the exact LEGAL_40K tokenizer...")
tok_candidates=[]
for base in (Path("/kaggle/input"), AUTO_EXTRACT_DIR):
    if base.exists():
        tok_candidates.extend(iter_files_pruned(base, names={"tokenizer.json","legal_40k_tokenizer.json"}))
matching=[]
for p in sorted(set(tok_candidates),key=lambda x:(len(str(x)),str(x))):
    try:
        if sha256_file(p)==TOKENIZER_SHA256: matching.append(p)
    except Exception: pass
# Do not recursively scan every JSON file as a fallback; that re-traverses the entire legal token-cache tree.

if not matching:
    print("ERROR: no attached tokenizer matches the official LEGAL_40K SHA-256:", TOKENIZER_SHA256)
    print("Tokenizer candidates found:")
    for q in sorted(set(tok_candidates), key=lambda p:(len(str(p)),str(p)))[:30]:
        try: print(" -", q, "| vocab-file-sha256=", sha256_file(q), "| bytes=", q.stat().st_size)
        except Exception as e: print(" -", q, "|", type(e).__name__, str(e)[:100])
    raise RuntimeError("Attach the exact Phase-1 LEGAL_40K tokenizer file matching the fingerprint; do not substitute a different vocabulary.")
TOKENIZER_PATH=sorted(matching,key=lambda p:(len(str(p)),str(p)))[0]
TOKENIZER=Tokenizer.from_file(str(TOKENIZER_PATH))
print("Tokenizer:",TOKENIZER_PATH,"| vocab:",TOKENIZER.get_vocab_size(),"| sha256:",sha256_file(TOKENIZER_PATH))
assert sha256_file(TOKENIZER_PATH)==TOKENIZER_SHA256 and TOKENIZER.get_vocab_size()==40000

# Persist a recovery manifest so progress is inspectable after interruption.
manifest={"data_paths":{k:str(v) for k,v in DATA_PATHS.items()},"checkpoint_paths":{k:str(v) for k,v in CHECKPOINT_PATHS.items()},"available_architectures":AVAILABLE_ARCHS,"missing_architectures":missing_arch,"tokenizer_path":str(TOKENIZER_PATH),"tokenizer_sha256":sha256_file(TOKENIZER_PATH),"train_count":len(TRAIN_RECORDS),"dev_count":len(DEV_RECORDS),"test_count":len(TEST_RECORDS),"device":str(DEVICE),"seed":SEED,"input_inventory_csv":str(ROOT/"input_inventory.csv")}
(ROOT/"recovery_manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
print("RECOVERY DISCOVERY PASSED")

In [ ]:
# Namespaces are isolated because all four implementation files reuse names such as RMSNorm and SwiGLU.
ARCH_CLASS_NAMES = {"A":"LegalDecoderA", "B":"LegalHybrid150", "C":"DMLC150", "D":"LegalPrefixFormer145"}
ARCH_SOURCE = {}
ARCH_SOURCE['A'] = 'class RMSNorm(nn.Module):\n\n    def __init__(\n        self,\n        dim,\n        eps=1e-5\n    ):\n\n        super().__init__()\n\n        self.weight = nn.Parameter(\n            torch.ones(dim)\n        )\n\n        self.eps = eps\n\n    def forward(self, x):\n\n        inv_rms = torch.rsqrt(\n            x.pow(2).mean(\n                dim=-1,\n                keepdim=True\n            )\n            + self.eps\n        )\n\n        return (\n            x\n            * inv_rms\n            * self.weight\n        )\n\nclass RotaryEmbedding(nn.Module):\n\n    def __init__(\n        self,\n        dim,\n        max_seq_len,\n        theta=10000.0\n    ):\n\n        super().__init__()\n\n        if dim % 2:\n            raise ValueError(\n                "RoPE dimension must be even."\n            )\n\n        inv_freq = 1.0 / (\n            theta ** (\n                torch.arange(\n                    0,\n                    dim,\n                    2,\n                    dtype=torch.float32\n                )\n                / dim\n            )\n        )\n\n        positions = torch.arange(\n            max_seq_len,\n            dtype=torch.float32\n        )\n\n        freqs = torch.outer(\n            positions,\n            inv_freq\n        )\n\n        self.register_buffer(\n            "cos_cached",\n            torch.repeat_interleave(\n                freqs.cos(),\n                2,\n                dim=-1\n            ),\n            persistent=False\n        )\n\n        self.register_buffer(\n            "sin_cached",\n            torch.repeat_interleave(\n                freqs.sin(),\n                2,\n                dim=-1\n            ),\n            persistent=False\n        )\n\n    def forward(\n        self,\n        x,\n        position_offset=0\n    ):\n\n        seq_len = x.size(-2)\n\n        end = (\n            position_offset\n            + seq_len\n        )\n\n        if (\n            end\n            > self.cos_cached.size(0)\n        ):\n            raise ValueError(\n                "RoPE context exceeded."\n            )\n\n        cos = self.cos_cached[\n            position_offset:end\n        ].to(\n            device=x.device,\n            dtype=x.dtype\n        )\n\n        sin = self.sin_cached[\n            position_offset:end\n        ].to(\n            device=x.device,\n            dtype=x.dtype\n        )\n\n        cos = cos.view(\n            1,\n            1,\n            seq_len,\n            -1\n        )\n\n        sin = sin.view(\n            1,\n            1,\n            seq_len,\n            -1\n        )\n\n        even = x[..., 0::2]\n        odd = x[..., 1::2]\n\n        rotated = torch.stack(\n            [\n                -odd,\n                even\n            ],\n            dim=-1\n        ).flatten(-2)\n\n        return (\n            x * cos\n            + rotated * sin\n        )\n\nclass GQAAttention(nn.Module):\n\n    def __init__(\n        self,\n        cfg\n    ):\n\n        super().__init__()\n\n        d_model = cfg[\n            "d_model"\n        ]\n\n        n_heads = cfg[\n            "n_heads"\n        ]\n\n        n_kv_heads = cfg[\n            "n_kv_heads"\n        ]\n\n        head_dim = (\n            d_model // n_heads\n        )\n\n        self.n_heads = n_heads\n        self.n_kv_heads = n_kv_heads\n        self.head_dim = head_dim\n\n        self.group_size = (\n            n_heads\n            // n_kv_heads\n        )\n\n        self.q_proj = nn.Linear(\n            d_model,\n            n_heads * head_dim,\n            bias=cfg["use_bias"]\n        )\n\n        self.k_proj = nn.Linear(\n            d_model,\n            n_kv_heads * head_dim,\n            bias=cfg["use_bias"]\n        )\n\n        self.v_proj = nn.Linear(\n            d_model,\n            n_kv_heads * head_dim,\n            bias=cfg["use_bias"]\n        )\n\n        self.o_proj = nn.Linear(\n            d_model,\n            d_model,\n            bias=cfg["use_bias"]\n        )\n\n        self.rope = RotaryEmbedding(\n            head_dim,\n            cfg["context_length"],\n            cfg["rope_theta"]\n        )\n\n    def forward(\n        self,\n        x,\n        past_kv=None,\n        use_cache=False\n    ):\n\n        batch, seq_len, d_model = (\n            x.shape\n        )\n\n        past_len = (\n            0\n            if past_kv is None\n            else past_kv[0].size(2)\n        )\n\n        q = self.q_proj(\n            x\n        ).view(\n            batch,\n            seq_len,\n            self.n_heads,\n            self.head_dim\n        ).transpose(\n            1,\n            2\n        )\n\n        k = self.k_proj(\n            x\n        ).view(\n            batch,\n            seq_len,\n            self.n_kv_heads,\n            self.head_dim\n        ).transpose(\n            1,\n            2\n        )\n\n        v = self.v_proj(\n            x\n        ).view(\n            batch,\n            seq_len,\n            self.n_kv_heads,\n            self.head_dim\n        ).transpose(\n            1,\n            2\n        )\n\n        q = self.rope(\n            q,\n            past_len\n        )\n\n        k = self.rope(\n            k,\n            past_len\n        )\n\n        if past_kv is not None:\n\n            past_k, past_v = past_kv\n\n            k = torch.cat(\n                [past_k, k],\n                dim=2\n            )\n\n            v = torch.cat(\n                [past_v, v],\n                dim=2\n            )\n\n        if (\n            self.n_kv_heads\n            != self.n_heads\n        ):\n\n            k_attn = (\n                k.repeat_interleave(\n                    self.group_size,\n                    dim=1\n                )\n            )\n\n            v_attn = (\n                v.repeat_interleave(\n                    self.group_size,\n                    dim=1\n                )\n            )\n\n        else:\n\n            k_attn = k\n            v_attn = v\n\n        if past_kv is None:\n\n            y = F.scaled_dot_product_attention(\n                q,\n                k_attn,\n                v_attn,\n                attn_mask=None,\n                dropout_p=0.0,\n                is_causal=True,\n            )\n\n        else:\n\n            y = F.scaled_dot_product_attention(\n                q,\n                k_attn,\n                v_attn,\n                attn_mask=None,\n                dropout_p=0.0,\n                is_causal=False,\n            )\n\n        y = y.transpose(\n            1,\n            2\n        ).contiguous().view(\n            batch,\n            seq_len,\n            d_model\n        )\n\n        output = self.o_proj(\n            y\n        )\n\n        if use_cache:\n            return output, (k, v)\n\n        return output\n\nclass SwiGLU(nn.Module):\n\n    def __init__(\n        self,\n        cfg\n    ):\n\n        super().__init__()\n\n        d_model = cfg[\n            "d_model"\n        ]\n\n        ffn_dim = cfg[\n            "ffn_dim"\n        ]\n\n        self.gate_proj = nn.Linear(\n            d_model,\n            ffn_dim,\n            bias=cfg["use_bias"]\n        )\n\n        self.up_proj = nn.Linear(\n            d_model,\n            ffn_dim,\n            bias=cfg["use_bias"]\n        )\n\n        self.down_proj = nn.Linear(\n            ffn_dim,\n            d_model,\n            bias=cfg["use_bias"]\n        )\n\n    def forward(self, x):\n\n        return self.down_proj(\n            F.silu(\n                self.gate_proj(x)\n            )\n            * self.up_proj(x)\n        )\n\nclass DecoderBlock(nn.Module):\n\n    def __init__(\n        self,\n        cfg\n    ):\n\n        super().__init__()\n\n        d_model = cfg[\n            "d_model"\n        ]\n\n        self.norm1 = RMSNorm(\n            d_model,\n            cfg["norm_eps"]\n        )\n\n        self.attn = GQAAttention(\n            cfg\n        )\n\n        self.norm2 = RMSNorm(\n            d_model,\n            cfg["norm_eps"]\n        )\n\n        self.ffn = SwiGLU(\n            cfg\n        )\n\n    def forward(\n        self,\n        x,\n        past_kv=None,\n        use_cache=False\n    ):\n\n        if use_cache:\n\n            attn_out, present = (\n                self.attn(\n                    self.norm1(x),\n                    past_kv=past_kv,\n                    use_cache=True,\n                )\n            )\n\n            x = x + attn_out\n\n            x = x + self.ffn(\n                self.norm2(x)\n            )\n\n            return x, present\n\n        x = x + self.attn(\n            self.norm1(x)\n        )\n\n        x = x + self.ffn(\n            self.norm2(x)\n        )\n\n        return x\n\nclass LegalDecoderA(nn.Module):\n\n    def __init__(\n        self,\n        cfg\n    ):\n\n        super().__init__()\n\n        self.cfg = dict(cfg)\n\n        self.token_embedding = nn.Embedding(\n            cfg["vocab_size"],\n            cfg["d_model"]\n        )\n\n        self.blocks = nn.ModuleList([\n            DecoderBlock(cfg)\n            for _ in range(\n                cfg["n_layers"]\n            )\n        ])\n\n        self.final_norm = RMSNorm(\n            cfg["d_model"],\n            cfg["norm_eps"]\n        )\n\n        self.lm_head = nn.Linear(\n            cfg["d_model"],\n            cfg["vocab_size"],\n            bias=False\n        )\n\n        if cfg["tie_embeddings"]:\n\n            self.lm_head.weight = (\n                self.token_embedding.weight\n            )\n\n        self.apply(\n            self._init_weights\n        )\n\n        residual_scale = (\n            1.0\n            / math.sqrt(\n                2.0\n                * cfg["n_layers"]\n            )\n        )\n\n        for block in self.blocks:\n\n            nn.init.normal_(\n                block.attn.o_proj.weight,\n                mean=0.0,\n                std=0.02\n                * residual_scale,\n            )\n\n            nn.init.normal_(\n                block.ffn.down_proj.weight,\n                mean=0.0,\n                std=0.02\n                * residual_scale,\n            )\n\n    def _init_weights(\n        self,\n        module\n    ):\n\n        if isinstance(\n            module,\n            nn.Linear\n        ):\n\n            if module.bias is not None:\n                nn.init.zeros_(\n                    module.bias\n                )\n\n            nn.init.normal_(\n                module.weight,\n                mean=0.0,\n                std=0.02,\n            )\n\n        elif isinstance(\n            module,\n            nn.Embedding\n        ):\n\n            nn.init.normal_(\n                module.weight,\n                mean=0.0,\n                std=0.02\n            )\n\n        elif isinstance(\n            module,\n            RMSNorm\n        ):\n\n            nn.init.ones_(\n                module.weight\n            )\n\n    def forward(\n        self,\n        input_ids,\n        labels=None,\n        use_cache=False,\n        past_key_values=None\n    ):\n\n        x = self.token_embedding(\n            input_ids\n        )\n\n        if use_cache:\n\n            if past_key_values is None:\n\n                past_key_values = [\n                    None\n                    for _ in self.blocks\n                ]\n\n            presents = []\n\n            for block, past in zip(\n                self.blocks,\n                past_key_values\n            ):\n\n                x, present = block(\n                    x,\n                    past_kv=past,\n                    use_cache=True\n                )\n\n                presents.append(\n                    present\n                )\n\n            x = self.final_norm(\n                x\n            )\n\n            logits = self.lm_head(\n                x\n            )\n\n            return {\n                "logits": logits,\n                "loss": None,\n                "past_key_values": presents,\n            }\n\n        for block in self.blocks:\n\n            if (\n                self.training\n                and self.cfg[\n                    "gradient_checkpointing"\n                ]\n            ):\n\n                x = torch.utils.checkpoint.checkpoint(\n                    block,\n                    x,\n                    use_reentrant=False\n                )\n\n            else:\n\n                x = block(x)\n\n        x = self.final_norm(\n            x\n        )\n\n        logits = self.lm_head(\n            x\n        )\n\n        loss = None\n\n        if labels is not None:\n\n            loss = F.cross_entropy(\n                logits.reshape(\n                    -1,\n                    logits.size(-1)\n                ),\n                labels.reshape(\n                    -1\n                )\n            )\n\n        return {\n            "logits": logits,\n            "loss": loss,\n        }'
ARCH_SOURCE['B'] = 'class RMSNorm(nn.Module):\n    def __init__(self, dim, eps=1e-5):\n        super().__init__()\n        self.weight = nn.Parameter(torch.ones(dim))\n        self.eps = eps\n\n    def forward(self, x):\n        # Stable fp32 RMS computation, then explicitly restore the incoming dtype.\n        inv_rms = torch.rsqrt(\n            x.float().pow(2).mean(dim=-1, keepdim=True) + self.eps\n        )\n        scale = self.weight.to(dtype=x.dtype)\n        out = (x * inv_rms.to(dtype=x.dtype)) * scale\n        return out.to(dtype=x.dtype)\n\nclass RotaryEmbedding(nn.Module):\n    def __init__(self, dim, max_seq_len, theta=10_000.0):\n        super().__init__()\n        if dim % 2:\n            raise ValueError("RoPE dimension must be even.")\n        inv_freq = 1.0 / (\n            theta ** (\n                torch.arange(0, dim, 2, dtype=torch.float32) / dim\n            )\n        )\n        positions = torch.arange(max_seq_len, dtype=torch.float32)\n        freqs = torch.outer(positions, inv_freq)\n        emb = torch.cat([freqs, freqs], dim=-1)\n        self.register_buffer("cos_cached", emb.cos(), persistent=False)\n        self.register_buffer("sin_cached", emb.sin(), persistent=False)\n\n    def forward(self, x, start_pos=0):\n        seq_len = x.size(-2)\n        end = start_pos + seq_len\n        if end > self.cos_cached.size(0):\n            raise RuntimeError(\n                f"RoPE request {end} exceeds max context {self.cos_cached.size(0)}"\n            )\n        cos = self.cos_cached[start_pos:end].to(dtype=x.dtype)[None, None, :, :]\n        sin = self.sin_cached[start_pos:end].to(dtype=x.dtype)[None, None, :, :]\n        x1 = x[..., :x.size(-1)//2]\n        x2 = x[..., x.size(-1)//2:]\n        rotated = torch.cat([-x2, x1], dim=-1)\n        return x * cos + rotated * sin\n\nclass HybridGQAAttention(nn.Module):\n    def __init__(self, cfg, is_global):\n        super().__init__()\n        d = cfg["d_model"]\n        h = cfg["n_heads"]\n        kvh = cfg["n_kv_heads"]\n        hd = cfg["head_dim"]\n\n        if d != h * hd:\n            raise ValueError("d_model must equal n_heads * head_dim.")\n        if h % kvh != 0:\n            raise ValueError("n_heads must be divisible by n_kv_heads.")\n\n        self.d_model = d\n        self.n_heads = h\n        self.n_kv_heads = kvh\n        self.head_dim = hd\n        self.group_size = h // kvh\n\n        self.is_global = bool(is_global)\n        self.local_window = int(cfg["local_window"])\n        self.query_chunk = int(cfg["local_query_chunk"])\n\n        self.q_proj = nn.Linear(d, h * hd, bias=cfg["use_bias"])\n        self.k_proj = nn.Linear(d, kvh * hd, bias=cfg["use_bias"])\n        self.v_proj = nn.Linear(d, kvh * hd, bias=cfg["use_bias"])\n        self.o_proj = nn.Linear(d, d, bias=cfg["use_bias"])\n\n        self.rope = RotaryEmbedding(\n            hd,\n            cfg["context_length"],\n            cfg["rope_theta"],\n        )\n\n    def _expand_kv(self, k, v):\n        if self.n_kv_heads == self.n_heads:\n            return k, v\n        k = k.repeat_interleave(self.group_size, dim=1)\n        v = v.repeat_interleave(self.group_size, dim=1)\n        return k, v\n\n    def _local_attention_train(self, q, k_full, v_full, past_len):\n        q_len = q.size(-2)\n        outputs = []\n\n        for start in range(0, q_len, self.query_chunk):\n            end = min(q_len, start + self.query_chunk)\n            curr = end - start\n\n            abs_q_start = past_len + start\n            key_start_abs = max(0, abs_q_start - self.local_window + 1)\n            key_end_abs = past_len + end\n\n            k_slice = k_full[:, :, key_start_abs:key_end_abs, :]\n            v_slice = v_full[:, :, key_start_abs:key_end_abs, :]\n\n            k_len = k_slice.size(-2)\n\n            # Exact causal sliding-window mask for this chunk. Construct only\n            # [query_chunk, local_window] rather than a full [T,T] mask.\n            q_positions = torch.arange(\n                abs_q_start,\n                past_len + end,\n                device=q.device,\n                dtype=torch.long,\n            )[:, None]\n            k_positions = torch.arange(\n                key_start_abs,\n                key_end_abs,\n                device=q.device,\n                dtype=torch.long,\n            )[None, :]\n            mask = (\n                (k_positions <= q_positions)\n                & (k_positions >= (q_positions - self.local_window + 1))\n            )\n            assert mask.shape == (curr, k_len)\n\n            y = F.scaled_dot_product_attention(\n                q[:, :, start:end, :],\n                k_slice,\n                v_slice,\n                attn_mask=mask,\n                dropout_p=0.0,\n                is_causal=False,\n            )\n            outputs.append(y)\n\n        return torch.cat(outputs, dim=-2)\n\n    def _local_attention_cached(self, q, k_full, v_full):\n        # Intended generation path: q_len=1. All retained keys are historical\n        # tokens plus the current token, so no causal mask is needed.\n        keep = min(self.local_window, k_full.size(-2))\n        k_slice = k_full[:, :, -keep:, :]\n        v_slice = v_full[:, :, -keep:, :]\n        return F.scaled_dot_product_attention(\n            q,\n            k_slice,\n            v_slice,\n            attn_mask=None,\n            dropout_p=0.0,\n            is_causal=False,\n        )\n\n    def forward(self, x, past_kv=None, use_cache=False):\n        bsz, q_len, _ = x.shape\n        past_len = 0 if past_kv is None else past_kv[0].size(2)\n\n        if past_len + q_len > self.rope.cos_cached.size(0):\n            raise RuntimeError("Attention request exceeds context length.")\n\n        q = self.q_proj(x).view(\n            bsz, q_len, self.n_heads, self.head_dim\n        ).transpose(1, 2)\n\n        k = self.k_proj(x).view(\n            bsz, q_len, self.n_kv_heads, self.head_dim\n        ).transpose(1, 2)\n\n        v = self.v_proj(x).view(\n            bsz, q_len, self.n_kv_heads, self.head_dim\n        ).transpose(1, 2)\n\n        q = self.rope(q, past_len)\n        k = self.rope(k, past_len)\n\n        if past_kv is not None:\n            past_k, past_v = past_kv\n            k_full = torch.cat([past_k, k], dim=2)\n            v_full = torch.cat([past_v, v], dim=2)\n        else:\n            k_full = k\n            v_full = v\n\n        if self.n_kv_heads != self.n_heads:\n            k_attn, v_attn = self._expand_kv(k_full, v_full)\n        else:\n            k_attn, v_attn = k_full, v_full\n\n        if self.is_global:\n            if past_kv is None:\n                y = F.scaled_dot_product_attention(\n                    q,\n                    k_attn,\n                    v_attn,\n                    attn_mask=None,\n                    dropout_p=0.0,\n                    is_causal=True,\n                )\n            else:\n                if q_len != 1:\n                    raise RuntimeError(\n                        "Cached global attention currently expects q_len=1."\n                    )\n                y = F.scaled_dot_product_attention(\n                    q,\n                    k_attn,\n                    v_attn,\n                    attn_mask=None,\n                    dropout_p=0.0,\n                    is_causal=False,\n                )\n        else:\n            if past_kv is not None:\n                if q_len != 1:\n                    raise RuntimeError(\n                        "Cached local attention currently expects q_len=1."\n                    )\n                y = self._local_attention_cached(q, k_attn, v_attn)\n            else:\n                y = self._local_attention_train(\n                    q, k_attn, v_attn, past_len=0\n                )\n\n        y = y.transpose(1, 2).contiguous().view(\n            bsz, q_len, self.d_model\n        )\n        y = self.o_proj(y)\n\n        if use_cache:\n            return y, (k_full, v_full)\n        return y\n\nclass SwiGLU(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        d = cfg["d_model"]\n        f = cfg["ffn_dim"]\n        self.gate_proj = nn.Linear(d, f, bias=cfg["use_bias"])\n        self.up_proj = nn.Linear(d, f, bias=cfg["use_bias"])\n        self.down_proj = nn.Linear(f, d, bias=cfg["use_bias"])\n\n    def forward(self, x):\n        return self.down_proj(\n            F.silu(self.gate_proj(x)) * self.up_proj(x)\n        )\n\nclass HybridBlock(nn.Module):\n    def __init__(self, cfg, layer_index):\n        super().__init__()\n        self.norm1 = RMSNorm(cfg["d_model"], cfg["norm_eps"])\n        self.attn = HybridGQAAttention(\n            cfg,\n            is_global=((layer_index + 1) % cfg["global_every"] == 0),\n        )\n        self.norm2 = RMSNorm(cfg["d_model"], cfg["norm_eps"])\n        self.ffn = SwiGLU(cfg)\n\n    def forward(self, x, past_kv=None, use_cache=False):\n        if use_cache:\n            attn_out, present = self.attn(\n                self.norm1(x),\n                past_kv=past_kv,\n                use_cache=True,\n            )\n            x = x + attn_out\n            x = x + self.ffn(self.norm2(x))\n            return x, present\n\n        x = x + self.attn(self.norm1(x))\n        x = x + self.ffn(self.norm2(x))\n        return x\n\nclass LegalHybrid150(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.cfg = dict(cfg)\n\n        self.token_embedding = nn.Embedding(\n            cfg["vocab_size"], cfg["d_model"]\n        )\n        self.blocks = nn.ModuleList([\n            HybridBlock(cfg, i)\n            for i in range(cfg["n_layers"])\n        ])\n        self.final_norm = RMSNorm(\n            cfg["d_model"], cfg["norm_eps"]\n        )\n        self.lm_head = nn.Linear(\n            cfg["d_model"],\n            cfg["vocab_size"],\n            bias=False,\n        )\n\n        # Initialize parameters once, then tie the LM head.\n        # This avoids double-initializing the same shared Parameter.\n        self.apply(self._init_weights)\n\n        if cfg["tie_embeddings"]:\n            self.lm_head.weight = self.token_embedding.weight\n\n        residual_scale = 1.0 / math.sqrt(2.0 * cfg["n_layers"])\n        for block in self.blocks:\n            nn.init.normal_(\n                block.attn.o_proj.weight,\n                mean=0.0,\n                std=0.02 * residual_scale,\n            )\n            nn.init.normal_(\n                block.ffn.down_proj.weight,\n                mean=0.0,\n                std=0.02 * residual_scale,\n            )\n\n    def _init_weights(self, module):\n        if isinstance(module, nn.Linear):\n            if module.bias is not None:\n                nn.init.zeros_(module.bias)\n            nn.init.normal_(module.weight, mean=0.0, std=0.02)\n        elif isinstance(module, nn.Embedding):\n            nn.init.normal_(module.weight, mean=0.0, std=0.02)\n        elif isinstance(module, RMSNorm):\n            nn.init.ones_(module.weight)\n\n    def forward(\n        self,\n        input_ids,\n        labels=None,\n        use_cache=False,\n        past_key_values=None,\n    ):\n        input_ids = input_ids.long()\n        x = self.token_embedding(input_ids)\n        # Keep the residual stream in FP16 on CUDA under the training/inference\n        # autocast path; this avoids unintended FP32 residual accumulation.\n        if x.is_cuda and x.dtype != torch.float16:\n            x = x.to(dtype=torch.float16)\n\n        if use_cache:\n            if past_key_values is None:\n                past_key_values = [None] * len(self.blocks)\n\n            presents = []\n            for block, past in zip(self.blocks, past_key_values):\n                x, present = block(\n                    x,\n                    past_kv=past,\n                    use_cache=True,\n                )\n                presents.append(present)\n\n            x = self.final_norm(x)\n            logits = self.lm_head(x)\n\n            return {\n                "logits": logits,\n                "loss": None,\n                "past_key_values": presents,\n            }\n\n        for block in self.blocks:\n            if (\n                self.training\n                and self.cfg["gradient_checkpointing"]\n            ):\n                x = torch.utils.checkpoint.checkpoint(\n                    block,\n                    x,\n                    use_reentrant=False,\n                )\n            else:\n                x = block(x)\n\n        x = self.final_norm(x)\n\n        if labels is not None:\n            # Keep peak memory reasonable by projecting in time chunks.\n            labels = labels.long()\n            time_chunk = 512\n            losses = []\n            for start in range(0, x.size(1), time_chunk):\n                end = min(x.size(1), start + time_chunk)\n                logits_chunk = self.lm_head(x[:, start:end, :])\n                loss_chunk = F.cross_entropy(\n                    logits_chunk.reshape(-1, logits_chunk.size(-1)),\n                    labels[:, start:end].reshape(-1),\n                )\n                losses.append(loss_chunk)\n                del logits_chunk\n            loss = torch.stack(losses).mean()\n            return {"loss": loss}\n\n        logits = self.lm_head(x)\n        return {"logits": logits, "loss": None}\n\ndef estimate_model_params(cfg):\n    v = cfg["vocab_size"]\n    d = cfg["d_model"]\n    L = cfg["n_layers"]\n    H = cfg["n_heads"]\n    KV = cfg["n_kv_heads"]\n    hd = cfg["head_dim"]\n    f = cfg["ffn_dim"]\n\n    embedding = v * d\n    attention = (\n        d * H * hd\n        + d * KV * hd\n        + d * KV * hd\n        + (H * hd) * d\n    )\n    ffn = 3 * d * f\n    norms = (2 * L + 1) * d\n    output_head = 0 if cfg["tie_embeddings"] else v * d\n\n    return embedding + L * (attention + ffn) + norms + output_head'
ARCH_SOURCE['C'] = 'class RMSNorm(nn.Module):\n    def __init__(self, dim, eps=1e-5):\n        super().__init__()\n        self.weight = nn.Parameter(torch.ones(dim))\n        self.eps = eps\n\n    def forward(self, x):\n        inv_rms = torch.rsqrt(x.float().pow(2).mean(dim=-1, keepdim=True) + self.eps)\n        return x * inv_rms.to(dtype=x.dtype) * self.weight.to(dtype=x.dtype)\n\nclass RotaryEmbedding(nn.Module):\n    def __init__(self, dim, max_seq_len, theta=10_000.0):\n        super().__init__()\n        if dim % 2:\n            raise ValueError("RoPE dimension must be even.")\n        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))\n        positions = torch.arange(max_seq_len, dtype=torch.float32)\n        freqs = torch.outer(positions, inv_freq)\n        emb = torch.cat([freqs, freqs], dim=-1)\n        self.register_buffer("cos_cached", emb.cos(), persistent=False)\n        self.register_buffer("sin_cached", emb.sin(), persistent=False)\n\n    def get(self, start, length, device, dtype):\n        if start + length > self.cos_cached.size(0):\n            raise RuntimeError("Requested RoPE positions exceed context length.")\n        cos = self.cos_cached[start:start+length].to(device=device, dtype=dtype).view(1, 1, length, -1)\n        sin = self.sin_cached[start:start+length].to(device=device, dtype=dtype).view(1, 1, length, -1)\n        return cos, sin\n\ndef rotate_half(x):\n    d = x.size(-1)\n    return torch.cat((-x[..., d//2:], x[..., :d//2]), dim=-1)\n\ndef apply_rope(x, cos, sin):\n    return x * cos + rotate_half(x) * sin\n\nclass GQAAttention(nn.Module):\n    def __init__(self, d_model, q_heads, kv_heads, head_dim, max_seq_len, theta=10_000.0, causal=False, use_rope=True):\n        super().__init__()\n        if d_model != q_heads * head_dim:\n            raise ValueError("d_model must equal q_heads * head_dim")\n        if q_heads % kv_heads:\n            raise ValueError("q_heads must be divisible by kv_heads")\n        self.q_heads = q_heads\n        self.kv_heads = kv_heads\n        self.head_dim = head_dim\n        self.causal = causal\n        self.use_rope = use_rope\n        self.q_proj = nn.Linear(d_model, q_heads * head_dim, bias=False)\n        self.k_proj = nn.Linear(d_model, kv_heads * head_dim, bias=False)\n        self.v_proj = nn.Linear(d_model, kv_heads * head_dim, bias=False)\n        self.o_proj = nn.Linear(q_heads * head_dim, d_model, bias=False)\n        self.rope = RotaryEmbedding(head_dim, max_seq_len, theta) if use_rope else None\n\n    def _q(self, x):\n        return self.q_proj(x).view(x.size(0), x.size(1), self.q_heads, self.head_dim).transpose(1, 2)\n\n    def _kv(self, x):\n        k = self.k_proj(x).view(x.size(0), x.size(1), self.kv_heads, self.head_dim).transpose(1, 2)\n        v = self.v_proj(x).view(x.size(0), x.size(1), self.kv_heads, self.head_dim).transpose(1, 2)\n        return k, v\n\n    def _repeat_kv(self, x):\n        return x.repeat_interleave(self.q_heads // self.kv_heads, dim=1)\n\n    @staticmethod\n    def _mask(mask):\n        if mask is None:\n            return None\n        if mask.dim() == 2:\n            return mask[:, None, None, :]\n        if mask.dim() == 3:\n            return mask[:, None, :, :]\n        return mask\n\n    def forward(self, q_input, kv_input=None, attn_mask=None, cache=None, use_cache=False, position_offset=0):\n        is_cross = kv_input is not None\n        source = q_input if kv_input is None else kv_input\n        q = self._q(q_input)\n        k_new, v_new = self._kv(source)\n\n        if self.use_rope and not is_cross:\n            cos_k, sin_k = self.rope.get(position_offset, k_new.size(2), k_new.device, k_new.dtype)\n            cos_q, sin_q = self.rope.get(position_offset, q.size(2), q.device, q.dtype)\n            k_new = apply_rope(k_new, cos_k, sin_k)\n            q = apply_rope(q, cos_q, sin_q)\n\n        if cache is not None:\n            k_cache = torch.cat([cache[0], k_new], dim=2)\n            v_cache = torch.cat([cache[1], v_new], dim=2)\n        else:\n            k_cache, v_cache = k_new, v_new\n\n        k = self._repeat_kv(k_cache)\n        v = self._repeat_kv(v_cache)\n        mask = self._mask(attn_mask)\n\n        if self.causal and cache is None and mask is None:\n            out = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, is_causal=True)\n        else:\n            out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=0.0, is_causal=False)\n\n        out = out.transpose(1, 2).contiguous().view(q_input.size(0), q_input.size(1), -1)\n        out = self.o_proj(out)\n        new_cache = (k_cache, v_cache) if use_cache else None\n        return out, new_cache\n\nclass SwiGLU(nn.Module):\n    def __init__(self, d_model, ffn_dim):\n        super().__init__()\n        self.gate_proj = nn.Linear(d_model, ffn_dim, bias=False)\n        self.up_proj = nn.Linear(d_model, ffn_dim, bias=False)\n        self.down_proj = nn.Linear(ffn_dim, d_model, bias=False)\n\n    def forward(self, x):\n        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))\n\nclass EncoderBlock(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        d = cfg["d_model"]\n        self.norm1 = RMSNorm(d, cfg["norm_eps"])\n        self.self_attn = GQAAttention(d, cfg["encoder_heads"], cfg["encoder_heads"], cfg["head_dim"], cfg["context_length"], cfg["rope_theta"], causal=False, use_rope=True)\n        self.norm2 = RMSNorm(d, cfg["norm_eps"])\n        self.ffn = SwiGLU(d, cfg["ffn_dim"])\n\n    def forward(self, x, attention_mask):\n        x = x + self.self_attn(self.norm1(x), attn_mask=attention_mask)[0]\n        x = x + self.ffn(self.norm2(x))\n        return x\n\nclass DecoderBlock(nn.Module):\n    def __init__(self, cfg, fusion=False):\n        super().__init__()\n        d = cfg["d_model"]\n        self.fusion = bool(fusion)\n        self.norm1 = RMSNorm(d, cfg["norm_eps"])\n        self.self_attn = GQAAttention(d, cfg["decoder_q_heads"], cfg["decoder_kv_heads"], cfg["head_dim"], cfg["context_length"], cfg["rope_theta"], causal=True, use_rope=True)\n        if self.fusion:\n            self.cross_norm = RMSNorm(d, cfg["norm_eps"])\n            self.case_cross_attn = GQAAttention(d, cfg["decoder_q_heads"], cfg["decoder_kv_heads"], cfg["head_dim"], cfg["context_length"], cfg["rope_theta"], causal=False, use_rope=False)\n            self.evidence_cross_attn = GQAAttention(d, cfg["decoder_q_heads"], cfg["decoder_kv_heads"], cfg["head_dim"], cfg["context_length"], cfg["rope_theta"], causal=False, use_rope=False)\n            gh = cfg["gate_hidden_dim"]\n            self.gate = nn.Sequential(nn.Linear(d, gh, bias=True), nn.Linear(gh, 1, bias=True))\n        self.norm2 = RMSNorm(d, cfg["norm_eps"])\n        self.ffn = SwiGLU(d, cfg["ffn_dim"])\n\n    def forward_full(self, x, evidence, case, evidence_mask, decoder_mask):\n        x = x + self.self_attn(self.norm1(x), attn_mask=decoder_mask)[0]\n        if self.fusion:\n            u = self.cross_norm(x)\n            case_out = self.case_cross_attn(u, case)[0]\n            evidence_out = self.evidence_cross_attn(u, evidence, attn_mask=evidence_mask)[0]\n            gate = torch.sigmoid(self.gate(u))\n            x = x + gate * evidence_out + (1.0 - gate) * case_out\n        x = x + self.ffn(self.norm2(x))\n        return x\n\n    def forward_step(self, x, evidence, case, evidence_mask, cache, position):\n        y, new_cache = self.self_attn(self.norm1(x), cache=cache, use_cache=True, position_offset=position)\n        x = x + y\n        if self.fusion:\n            u = self.cross_norm(x)\n            case_out = self.case_cross_attn(u, case)[0]\n            evidence_out = self.evidence_cross_attn(u, evidence, attn_mask=evidence_mask)[0]\n            gate = torch.sigmoid(self.gate(u))\n            x = x + gate * evidence_out + (1.0 - gate) * case_out\n        x = x + self.ffn(self.norm2(x))\n        return x, new_cache\n\nclass DMLC150(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.cfg = dict(cfg)\n        d = int(cfg["d_model"])\n        V = int(cfg["vocab_size"])\n        self.token_embedding = nn.Embedding(V, d)\n        self.encoder = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg["encoder_layers"])])\n        self.encoder_final_norm = RMSNorm(d, cfg["norm_eps"])\n        self.case_memory = nn.Parameter(torch.empty(cfg["case_memory_slots"], d))\n        self.case_memory_slots = int(cfg["case_memory_slots"])\n        self.case_memory_attention = GQAAttention(d, cfg["decoder_q_heads"], cfg["decoder_kv_heads"], cfg["head_dim"], cfg["context_length"], cfg["rope_theta"], causal=False, use_rope=False)\n        self.case_memory_norm = RMSNorm(d, cfg["norm_eps"])\n        fusion = set(int(x) for x in cfg["memory_fusion_layers"])\n        self.decoder = nn.ModuleList([DecoderBlock(cfg, i in fusion) for i in range(1, cfg["decoder_layers"] + 1)])\n        self.fusion_layers = nn.ModuleDict({str(i): self.decoder[i - 1] for i in sorted(fusion)})\n        self.decoder_final_norm = RMSNorm(d, cfg["norm_eps"])\n        self.decoder_start = nn.Parameter(torch.empty(d))\n        self.lm_head = nn.Linear(d, V, bias=False)\n        self.lm_head.weight = self.token_embedding.weight\n        self._reset_parameters(cfg)\n\n    def _reset_parameters(self, cfg):\n        total_residual_layers = cfg["encoder_layers"] + cfg["decoder_layers"]\n        small_std = 0.02 / math.sqrt(max(1.0, 2.0 * total_residual_layers))\n        for name, module in self.named_modules():\n            if isinstance(module, nn.Linear):\n                use_small = name.endswith("o_proj") or name.endswith("down_proj")\n                std = small_std if use_small else 0.02\n                nn.init.normal_(module.weight, mean=0.0, std=std)\n                if module.bias is not None:\n                    nn.init.zeros_(module.bias)\n            elif isinstance(module, nn.Embedding):\n                nn.init.normal_(module.weight, mean=0.0, std=0.02)\n            elif isinstance(module, RMSNorm):\n                nn.init.ones_(module.weight)\n        nn.init.normal_(self.case_memory, mean=0.0, std=0.02)\n        nn.init.normal_(self.decoder_start, mean=0.0, std=0.02)\n        # Start with an unbiased 50/50 evidence-vs-case mixture.\n        for layer in self.fusion_layers.values():\n            nn.init.zeros_(layer.gate[-1].weight)\n            nn.init.zeros_(layer.gate[-1].bias)\n\n    def _encode(self, src_ids, src_attention_mask):\n        x = self.token_embedding(src_ids)\n        for layer in self.encoder:\n            if self.training and self.cfg.get("gradient_checkpointing", True):\n                x = checkpoint(layer, x, src_attention_mask, use_reentrant=False)\n            else:\n                x = layer(x, src_attention_mask)\n        evidence = self.encoder_final_norm(x)\n        case_seed = self.case_memory.unsqueeze(0).expand(src_ids.size(0), -1, -1)\n        case = case_seed + self.case_memory_attention(case_seed, evidence, attn_mask=src_attention_mask)[0]\n        case = self.case_memory_norm(case)\n        return evidence, case\n\n    def encode_prompt(self, src_ids, src_attention_mask=None):\n        if src_ids.dim() != 2:\n            raise ValueError("src_ids must be [B,S].")\n        if src_ids.size(1) > self.cfg["context_length"]:\n            raise RuntimeError("Source sequence exceeds context length.")\n        evidence, case = self._encode(src_ids, src_attention_mask)\n        return {"evidence_memory": evidence, "case_memory": case, "src_attention_mask": src_attention_mask}\n\n    def decode_teacher_forced(self, decoder_input_ids, memory, decoder_attention_mask=None):\n        if decoder_input_ids.dim() != 2:\n            raise ValueError("decoder_input_ids must be [B,T].")\n        B, T = decoder_input_ids.shape\n        if T > self.cfg["context_length"]:\n            raise RuntimeError("Target sequence exceeds decoder context length.")\n        start = self.decoder_start.view(1, 1, -1).expand(B, 1, -1)\n        shifted = self.token_embedding(decoder_input_ids[:, :-1]) if T > 1 else decoder_input_ids.new_empty((B, 0, self.cfg["d_model"]), dtype=torch.float32)\n        x = torch.cat([start, shifted], dim=1)\n        T2 = x.size(1)\n        if decoder_attention_mask is None:\n            causal = None  # Let SDPA use its fused causal path.\n        else:\n            causal = torch.tril(torch.ones((T2, T2), dtype=torch.bool, device=x.device))[None, None]\n            causal = causal & decoder_attention_mask[:, None, None, :]\n        evidence = memory["evidence_memory"]\n        case = memory["case_memory"]\n        evidence_mask = memory["src_attention_mask"]\n        for i, layer in enumerate(self.decoder, start=1):\n            if self.training and self.cfg.get("gradient_checkpointing", True):\n                x = checkpoint(layer.forward_full, x, evidence, case, evidence_mask, causal, use_reentrant=False)\n            else:\n                x = layer.forward_full(x, evidence, case, evidence_mask, causal)\n        return self.decoder_final_norm(x)\n\n    def forward(self, src_ids, decoder_input_ids=None, labels=None, src_attention_mask=None, decoder_attention_mask=None, return_logits=True):\n        if labels is not None and decoder_input_ids is None:\n            decoder_input_ids = labels\n        if decoder_input_ids is None:\n            raise ValueError("Provide decoder_input_ids or labels.")\n        memory = self.encode_prompt(src_ids, src_attention_mask)\n        hidden = self.decode_teacher_forced(decoder_input_ids, memory, decoder_attention_mask)\n        logits = F.linear(hidden, self.token_embedding.weight)\n        out = {}\n        if labels is not None:\n            out["loss"] = F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1), ignore_index=-100)\n        if return_logits:\n            out["logits"] = logits\n        return out\n\n    @torch.no_grad()\n    def generate(self, src_ids, src_attention_mask=None, max_new_tokens=512, temperature=0.10, top_p=0.95, top_k=20, special_token_ids=None):\n        if special_token_ids is None:\n            special_token_ids = set()\n        memory = self.encode_prompt(src_ids, src_attention_mask)\n        B = src_ids.size(0)\n        x = self.decoder_start.view(1, 1, -1).expand(B, 1, -1)\n        caches = [None] * len(self.decoder)\n        generated = []\n        for position in range(int(max_new_tokens)):\n            for i, layer in enumerate(self.decoder):\n                x, caches[i] = layer.forward_step(x, memory["evidence_memory"], memory["case_memory"], memory["src_attention_mask"], caches[i], position)\n            logits = F.linear(self.decoder_final_norm(x)[:, -1], self.token_embedding.weight).float()\n            for token_id in special_token_ids:\n                if 0 <= int(token_id) < logits.size(-1):\n                    logits[:, int(token_id)] = -float("inf")\n            if temperature is None or float(temperature) <= 0:\n                next_ids = torch.argmax(logits, dim=-1)\n            else:\n                logits = logits / max(float(temperature), 1e-5)\n                if top_k and 0 < int(top_k) < logits.size(-1):\n                    values, _ = torch.topk(logits, int(top_k), dim=-1)\n                    cutoff = values[:, -1, None]\n                    logits = torch.where(logits < cutoff, torch.full_like(logits, -float("inf")), logits)\n                if top_p is not None and 0 < float(top_p) < 1:\n                    sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)\n                    probs = torch.softmax(sorted_logits, dim=-1)\n                    cumulative = torch.cumsum(probs, dim=-1)\n                    remove = cumulative > float(top_p)\n                    remove[:, 1:] = remove[:, :-1].clone()\n                    remove[:, 0] = False\n                    sorted_logits = sorted_logits.masked_fill(remove, -float("inf"))\n                    logits = torch.full_like(logits, -float("inf")).scatter(1, sorted_indices, sorted_logits)\n                next_ids = torch.multinomial(torch.softmax(logits, dim=-1), 1).squeeze(-1)\n            generated.append(next_ids)\n            x = self.token_embedding(next_ids).unsqueeze(1)\n        return torch.stack(generated, dim=1) if generated else src_ids.new_empty((B, 0))'
ARCH_SOURCE['D'] = 'class RMSNorm(nn.Module):\n    def __init__(self, dim, eps=1e-5):\n        super().__init__()\n        self.weight = nn.Parameter(torch.ones(dim))\n        self.eps = eps\n\n    def forward(self, x):\n        rms = torch.rsqrt(\n            x.float().pow(2).mean(dim=-1, keepdim=True) + self.eps\n        ).to(x.dtype)\n        return x * rms * self.weight.to(x.dtype)\n\nclass RoPE(nn.Module):\n    def __init__(self, head_dim, max_len, theta=10000.0):\n        super().__init__()\n        inv_freq = 1.0 / (\n            theta ** (\n                torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim\n            )\n        )\n        pos = torch.arange(max_len, dtype=torch.float32)\n        freqs = torch.outer(pos, inv_freq)\n        self.register_buffer("cos", freqs.cos(), persistent=False)\n        self.register_buffer("sin", freqs.sin(), persistent=False)\n\n    def rotate(self, x, offset=0):\n        t = x.size(-2)\n        c = self.cos[offset:offset+t].to(x.device, x.dtype)[None, None, :, :]\n        s = self.sin[offset:offset+t].to(x.device, x.dtype)[None, None, :, :]\n        xe, xo = x[..., 0::2], x[..., 1::2]\n        y = torch.empty_like(x)\n        y[..., 0::2] = xe * c - xo * s\n        y[..., 1::2] = xe * s + xo * c\n        return y\n\nclass PrefixGQAAttention(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        d = cfg["d_model"]\n        H = cfg["n_heads"]\n        KV = cfg["n_kv_heads"]\n        hd = cfg["head_dim"]\n\n        self.H = H\n        self.KV = KV\n        self.hd = hd\n        self.group = H // KV\n\n        self.q_proj = nn.Linear(d, H * hd, bias=False)\n        self.k_proj = nn.Linear(d, KV * hd, bias=False)\n        self.v_proj = nn.Linear(d, KV * hd, bias=False)\n        self.o_proj = nn.Linear(d, d, bias=False)\n\n        self.rope = RoPE(hd, cfg["context_length"], cfg["rope_theta"])\n        self.q_norm = RMSNorm(hd, cfg["norm_eps"]) if cfg.get("qk_norm") else nn.Identity()\n        self.k_norm = RMSNorm(hd, cfg["norm_eps"]) if cfg.get("qk_norm") else nn.Identity()\n\n        suffix_len = cfg["context_length"] - cfg["prefix_length"]\n        prefix_len = cfg["prefix_length"]\n\n        # Small suffix mask: [suffix_queries, prefix+suffix_keys].\n        mask = torch.ones(\n            suffix_len, prefix_len + suffix_len, dtype=torch.bool\n        )\n        mask[:, prefix_len:] = torch.tril(\n            torch.ones(suffix_len, suffix_len, dtype=torch.bool)\n        )\n        self.register_buffer(\n            "suffix_mask",\n            mask[None, None, :, :],\n            persistent=False,\n        )\n\n    def forward(self, x, prefix_len, past_kv=None, use_cache=False):\n        B, T, D = x.shape\n        past_len = 0 if past_kv is None else past_kv[0].size(2)\n\n        q = self.q_proj(x).view(B, T, self.H, self.hd).transpose(1, 2)\n        k = self.k_proj(x).view(B, T, self.KV, self.hd).transpose(1, 2)\n        v = self.v_proj(x).view(B, T, self.KV, self.hd).transpose(1, 2)\n\n        q = self.q_norm(self.rope.rotate(q, past_len))\n        k = self.k_norm(self.rope.rotate(k, past_len))\n\n        if past_kv is not None:\n            pk, pv = past_kv\n            k = torch.cat([pk, k], dim=2)\n            v = torch.cat([pv, v], dim=2)\n\n        ka = k.repeat_interleave(self.group, dim=1)\n        va = v.repeat_interleave(self.group, dim=1)\n\n        if past_kv is not None:\n            y = F.scaled_dot_product_attention(\n                q, ka, va,\n                attn_mask=None,\n                dropout_p=0.0,\n                is_causal=False,\n            )\n        else:\n            p = int(prefix_len)\n            q_pref, q_suf = q[:, :, :p], q[:, :, p:]\n            k_all, v_all = ka, va\n\n            y_pref = F.scaled_dot_product_attention(\n                q_pref, k_all[:, :, :p], v_all[:, :, :p],\n                attn_mask=None,\n                dropout_p=0.0,\n                is_causal=False,\n            )\n\n            mask = self.suffix_mask[:, :, :q_suf.size(2), :k_all.size(2)]\n            y_suf = F.scaled_dot_product_attention(\n                q_suf, k_all, v_all,\n                attn_mask=mask,\n                dropout_p=0.0,\n                is_causal=False,\n            )\n\n            y = torch.cat([y_pref, y_suf], dim=2)\n\n        y = y.transpose(1, 2).contiguous().view(B, T, D)\n        out = self.o_proj(y)\n\n        return (out, (k, v)) if use_cache else out\n\nclass SwiGLU(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        d, f = cfg["d_model"], cfg["ffn_dim"]\n        self.gate = nn.Linear(d, f, bias=False)\n        self.up = nn.Linear(d, f, bias=False)\n        self.down = nn.Linear(f, d, bias=False)\n\n    def forward(self, x):\n        return self.down(F.silu(self.gate(x)) * self.up(x))\n\nclass PrefixBlock(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.norm1 = RMSNorm(cfg["d_model"], cfg["norm_eps"])\n        self.attn = PrefixGQAAttention(cfg)\n        self.norm2 = RMSNorm(cfg["d_model"], cfg["norm_eps"])\n        self.ffn = SwiGLU(cfg)\n\n    def forward(self, x, prefix_len, past_kv=None, use_cache=False):\n        if use_cache:\n            a, present = self.attn(\n                self.norm1(x),\n                prefix_len,\n                past_kv=past_kv,\n                use_cache=True,\n            )\n            x = x + a\n            x = x + self.ffn(self.norm2(x))\n            return x, present\n\n        x = x + self.attn(self.norm1(x), prefix_len)\n        x = x + self.ffn(self.norm2(x))\n        return x\n\nclass LegalPrefixFormer145(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.cfg = dict(cfg)\n        d = cfg["d_model"]\n\n        self.token_embedding = nn.Embedding(cfg["vocab_size"], d)\n        self.blocks = nn.ModuleList([\n            PrefixBlock(cfg) for _ in range(cfg["n_layers"])\n        ])\n        self.final_norm = RMSNorm(d, cfg["norm_eps"])\n        self.lm_head = nn.Linear(d, cfg["vocab_size"], bias=False)\n\n        if cfg["tie_embeddings"]:\n            self.lm_head.weight = self.token_embedding.weight\n\n        self.apply(self._init_weights)\n\n        residual_scale = 1.0 / math.sqrt(2.0 * cfg["n_layers"])\n        for block in self.blocks:\n            nn.init.normal_(\n                block.attn.o_proj.weight,\n                mean=0.0,\n                std=0.02 * residual_scale,\n            )\n            nn.init.normal_(\n                block.ffn.down.weight,\n                mean=0.0,\n                std=0.02 * residual_scale,\n            )\n\n    def _init_weights(self, module):\n        if isinstance(module, nn.Linear):\n            nn.init.normal_(module.weight, mean=0.0, std=0.02)\n            if module.bias is not None:\n                nn.init.zeros_(module.bias)\n        elif isinstance(module, nn.Embedding):\n            nn.init.normal_(module.weight, mean=0.0, std=0.02)\n\n    def _run_block(self, block, x, prefix_len):\n        return block(x, prefix_len)\n\n    def forward(\n        self,\n        input_ids,\n        labels=None,\n        prefix_len=None,\n        past_key_values=None,\n        use_cache=False,\n        return_logits=None,\n    ):\n        if prefix_len is None:\n            prefix_len = self.cfg["prefix_length"]\n\n        x = self.token_embedding(input_ids)\n        presents = []\n\n        for i, block in enumerate(self.blocks):\n            pkv = None if past_key_values is None else past_key_values[i]\n\n            if use_cache:\n                x, present = block(\n                    x,\n                    prefix_len,\n                    past_kv=pkv,\n                    use_cache=True,\n                )\n                presents.append(present)\n            elif self.training and self.cfg.get("activation_checkpointing", False):\n                x = activation_checkpoint(\n                    self._run_block,\n                    block,\n                    x,\n                    prefix_len,\n                    use_reentrant=False,\n                )\n            else:\n                x = block(x, prefix_len)\n\n        x = self.final_norm(x)\n\n        loss = None\n        logits = None\n\n        if labels is not None:\n            # Input layout:\n            # prefix[0:1024], BOS[1024], target-input[1025:1280].\n            # Prediction logits at positions [1024:1280] map to 256 labels.\n            target_hidden = x[:, prefix_len:, :]\n            target_logits = self.lm_head(target_hidden)\n            target_labels = labels[:, prefix_len:]\n            loss = F.cross_entropy(\n                target_logits.reshape(-1, target_logits.size(-1)).float(),\n                target_labels.reshape(-1),\n                ignore_index=-100,\n            )\n\n        if return_logits is None:\n            return_logits = labels is None or use_cache\n\n        if return_logits:\n            logits = self.lm_head(x)\n\n        return {\n            "logits": logits,\n            "loss": loss,\n            "past_key_values": tuple(presents) if use_cache else None,\n        }'
ARCH_NS = {}
for _arch, _source in ARCH_SOURCE.items():
    _ns = {"torch":torch, "nn":nn, "F":F, "math":math, "np":np, "checkpoint":checkpoint, "activation_checkpoint":checkpoint}
    exec(compile(_source, f'<phase1_arch_{_arch}>', 'exec'), _ns, _ns)
    if ARCH_CLASS_NAMES[_arch] not in _ns: raise RuntimeError(f'Missing class for architecture {_arch}')
    ARCH_NS[_arch] = _ns
    print(f'Architecture {_arch}: source classes loaded | {ARCH_CLASS_NAMES[_arch]}')
print('All four existing Phase-1 architecture implementations are ready.')

In [ ]:
# Full train/dev/test splits are retained. If the cell/runtime is interrupted, rerunning this cell resumes each split.
MAX_SEQ_LEN = 512
HEAD_TOKENS = MAX_SEQ_LEN // 2
TAIL_TOKENS = MAX_SEQ_LEN - HEAD_TOKENS
BATCH_SIZE = 2
EVAL_BATCH_SIZE = 4
CACHE_DIR = ROOT / "tokenized_cache_v2"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_CHUNK = 500

TRAIN_USED = list(TRAIN_RECORDS)
DEV_USED = list(DEV_RECORDS)

def _split_signature(records, labeled):
    h = hashlib.sha256()
    h.update(f"n={len(records)}|seq={MAX_SEQ_LEN}|tok={TOKENIZER_SHA256}|lab={int(labeled)}\n".encode())
    for r in records:
        h.update(str(r["id"]).encode("utf-8", "ignore")); h.update(b"\0")
        text = str(r.get("text", ""))
        h.update(hashlib.sha256(text.encode("utf-8", "ignore")).digest()); h.update(b"\0")
        if labeled: h.update(str(int(r["label"])).encode())
        h.update(b"\n")
    return h.hexdigest()

def _encode_one_record(record):
    text = record.get("text", "")
    ids = TOKENIZER.encode(text, add_special_tokens=False).ids
    if not ids: ids = [0]
    if len(ids) > MAX_SEQ_LEN:
        ids = ids[:HEAD_TOKENS] + ids[-TAIL_TOKENS:]
    valid = len(ids)
    if valid < MAX_SEQ_LEN:
        ids = ids + [0] * (MAX_SEQ_LEN - valid)
    mask = np.zeros(MAX_SEQ_LEN, dtype=np.bool_)
    mask[:valid] = True
    return np.asarray(ids, dtype=np.int64), mask

def encode_split_resumable(records, name, labeled):
    n = len(records)
    sig = _split_signature(records, labeled)
    ids_path = CACHE_DIR / f"{name}_ids.npy"
    mask_path = CACHE_DIR / f"{name}_mask.npy"
    state_path = CACHE_DIR / f"{name}_progress.json"
    labels = np.asarray([int(r["label"]) for r in records], dtype=np.int64) if labeled else None
    start = 0
    compatible = False
    if ids_path.exists() and mask_path.exists() and state_path.exists():
        try:
            st = json.loads(state_path.read_text(encoding="utf-8"))
            compatible = st.get("signature") == sig and st.get("count") == n and st.get("seq_len") == MAX_SEQ_LEN
            if compatible: start = int(st.get("next_index", 0))
        except Exception: compatible = False
    if not compatible:
        for p in (ids_path, mask_path, state_path):
            try: p.unlink()
            except FileNotFoundError: pass
        xmap = np.lib.format.open_memmap(ids_path, mode="w+", dtype=np.int64, shape=(n, MAX_SEQ_LEN))
        mmap = np.lib.format.open_memmap(mask_path, mode="w+", dtype=np.bool_, shape=(n, MAX_SEQ_LEN))
        start = 0
        state_path.write_text(json.dumps({"signature":sig,"count":n,"seq_len":MAX_SEQ_LEN,"next_index":0}), encoding="utf-8")
    else:
        xmap = np.lib.format.open_memmap(ids_path, mode="r+", dtype=np.int64, shape=(n, MAX_SEQ_LEN))
        mmap = np.lib.format.open_memmap(mask_path, mode="r+", dtype=np.bool_, shape=(n, MAX_SEQ_LEN))
    print(f"{name}: cache={'resume' if start else 'build'} | start={start:,}/{n:,}", flush=True)
    for i in range(start, n):
        x, m = _encode_one_record(records[i])
        xmap[i] = x; mmap[i] = m
        if (i + 1) % CACHE_CHUNK == 0 or i + 1 == n:
            xmap.flush(); mmap.flush()
            state_path.write_text(json.dumps({"signature":sig,"count":n,"seq_len":MAX_SEQ_LEN,"next_index":i+1}), encoding="utf-8")
            print(f"{name}: tokenized {i+1:,}/{n:,}", flush=True)
    # Copy-on-write views prevent accidental writes to cached arrays while keeping RAM usage low.
    xmap = np.load(ids_path, mmap_mode="c")
    mmap = np.load(mask_path, mmap_mode="c")
    X = torch.from_numpy(xmap)
    M = torch.from_numpy(mmap)
    Y = torch.from_numpy(labels) if labeled else None
    return X, M, Y

print("Official records:", len(TRAIN_USED), "train |", len(DEV_USED), "dev |", len(TEST_RECORDS), "test")
print("Train labels:", dict(Counter(int(r["label"]) for r in TRAIN_USED)))
print("Dev labels:", dict(Counter(int(r["label"]) for r in DEV_USED)))
TRAIN_X, TRAIN_M, TRAIN_Y = encode_split_resumable(TRAIN_USED, "train", True)
DEV_X, DEV_M, DEV_Y = encode_split_resumable(DEV_USED, "dev", True)
TEST_X, TEST_M, _ = encode_split_resumable(TEST_RECORDS, "test", False)
assert TRAIN_X.shape == (len(TRAIN_USED), MAX_SEQ_LEN)
assert DEV_X.shape == (len(DEV_USED), MAX_SEQ_LEN)
assert TEST_X.shape == (len(TEST_RECORDS), MAX_SEQ_LEN)
assert int(TRAIN_Y.min()) == 0 and int(TRAIN_Y.max()) == 1
DATA_SIGNATURE = hashlib.sha256((
    _split_signature(TRAIN_USED, True) + _split_signature(DEV_USED, True) + _split_signature(TEST_RECORDS, False)
).encode()).hexdigest()
(ROOT / "data_manifest.json").write_text(json.dumps({
    "train_count":len(TRAIN_USED),"dev_count":len(DEV_USED),"test_count":len(TEST_RECORDS),
    "seq_len":MAX_SEQ_LEN,"head_tokens":HEAD_TOKENS,"tail_tokens":TAIL_TOKENS,
    "data_signature":DATA_SIGNATURE,"tokenizer_sha256":TOKENIZER_SHA256,
    "cache_dir":str(CACHE_DIR),"train_id_duplicates":len(TRAIN_USED)-len(set(x["id"] for x in TRAIN_USED)),
    "dev_id_duplicates":len(DEV_USED)-len(set(x["id"] for x in DEV_USED))
},indent=2), encoding="utf-8")
print("TOKEN CACHE READY | shapes", tuple(TRAIN_X.shape), tuple(DEV_X.shape), tuple(TEST_X.shape))


In [ ]:
CLASS_NAMES = {"A":"LegalDecoderA","B":"LegalHybrid150","C":"DMLC150","D":"LegalPrefixFormer145"}

class HiddenCaptureHead(nn.Module):
    """Avoids allocating 40K logits; captures the Phase-1 backbone final hidden states."""
    def __init__(self):
        super().__init__()
        self.last_hidden=None
    def forward(self, hidden):
        self.last_hidden=hidden
        return hidden[..., :1] * 0.0

class BottleneckAdapter(nn.Module):
    def __init__(self, dim, bottleneck=32):
        super().__init__()
        self.down=nn.Linear(dim,bottleneck,bias=True)
        self.act=nn.GELU()
        self.up=nn.Linear(bottleneck,dim,bias=True)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self,x): return self.up(self.act(self.down(x)))

class LoRMALinear(nn.Module):
    """Output-space low-rank multiplicative adaptation: y' = y + (y A^T) B^T."""
    def __init__(self, base, rank=4, alpha=8.0):
        super().__init__(); self.base=base; self.rank=int(rank); self.alpha=float(alpha)
        for p in self.base.parameters(): p.requires_grad=False
        out_dim=base.out_features
        self.A=nn.Parameter(torch.empty(self.rank,out_dim))
        self.B=nn.Parameter(torch.zeros(out_dim,self.rank))
        nn.init.normal_(self.A,mean=0.0,std=0.01)
    def forward(self,x):
        y=self.base(x)
        return y + F.linear(F.linear(y,self.A),self.B)*(self.alpha/self.rank)

def set_submodule(root,name,module):
    bits=name.split("."); parent=root
    for b in bits[:-1]: parent=getattr(parent,b)
    setattr(parent,bits[-1],module)

def install_lorma(backbone,rank=4,alpha=8.0):
    # Select attention and FFN projections. The LM projection is not adapted because it is replaced by a hidden-state tap.
    targets=[]
    for name,module in list(backbone.named_modules()):
        if not name or not isinstance(module,nn.Linear): continue
        low=name.lower()
        if any(x in low for x in ("attn.","attention.","ffn.","gate.")) and not low.endswith("lm_head"):
            targets.append((name,module))
    for name,module in targets:
        set_submodule(backbone,name,LoRMALinear(module,rank=rank,alpha=alpha))
    if not targets: raise RuntimeError("LoRMA did not find any attention/FFN Linear projections")
    print("LoRMA wrapped linear projections:",len(targets))

def _layer_groups(backbone, arch):
    groups=[]
    if hasattr(backbone,"blocks") and isinstance(backbone.blocks,nn.ModuleList):
        groups.append(("blocks",list(backbone.blocks)))
    if arch=="C":
        if hasattr(backbone,"encoder"): groups.append(("encoder",list(backbone.encoder)))
        if hasattr(backbone,"decoder"): groups.append(("decoder",list(backbone.decoder)))
    return groups

def install_adapters(backbone, arch, bottleneck=32, last_n=4):
    groups=_layer_groups(backbone,arch)
    if not groups: raise RuntimeError(f"No supported existing layer list available for adapters on architecture {arch}")
    group_name,layers=groups[-1]
    indices=list(range(max(0,len(layers)-int(last_n)),len(layers)))
    d=int(backbone.cfg["d_model"])
    backbone.cjpe_adapter_bank=nn.ModuleDict({str(i):BottleneckAdapter(d,bottleneck) for i in indices})
    handles=[]
    for i in indices:
        def make_hook(key):
            def hook(module,inputs,output):
                adapter=backbone.cjpe_adapter_bank[key]
                if isinstance(output,tuple): return (output[0]+adapter(output[0]),)+output[1:]
                return output+adapter(output)
            return hook
        handles.append(layers[i].register_forward_hook(make_hook(str(i))))
    backbone._cjpe_hook_handles=handles
    backbone._cjpe_adapter_group=group_name
    print(f"Bottleneck adapters attached to existing {group_name} layers:",indices)

def get_payload_state(payload):
    for k in ("model_state_dict","state_dict","model_weights","weights","model"):
        v=payload.get(k) if isinstance(payload,dict) else None
        if isinstance(v,dict) and v and any(torch.is_tensor(z) for z in v.values()): return v
    raise RuntimeError("No model state dictionary in checkpoint")

def clean_state_keys(state):
    keys=list(state)
    for prefix in ("module.",):
        if keys and all(k.startswith(prefix) for k in keys):
            state={k[len(prefix):]:v for k,v in state.items()}; keys=list(state)
    return state

def load_backbone(arch, method, checkpoint_path=None):
    path=Path(checkpoint_path or CHECKPOINT_PATHS[arch])
    payload=torch.load(path,map_location="cpu",weights_only=False)
    cfg=payload_config(payload)
    if not cfg: cfg=CHECKPOINT_META[arch]["cfg"]
    state=clean_state_keys(get_payload_state(payload))
    cls=ARCH_NS[arch][CLASS_NAMES[arch]]
    backbone=cls(cfg)
    missing,unexpected=backbone.load_state_dict(state,strict=False)
    # Strict-ish compatibility check: tied lm_head aliases may be missing when a checkpoint only stores one alias.
    bad_missing=[x for x in missing if not (x.endswith("lm_head.weight") and cfg.get("tie_embeddings",False))]
    bad_unexpected=list(unexpected)
    if bad_missing or bad_unexpected:
        raise RuntimeError(f"Checkpoint/model mismatch for {arch}. missing={bad_missing[:8]} unexpected={bad_unexpected[:8]} path={path}")
    if arch in "ABD": backbone.lm_head=HiddenCaptureHead()
    # Configure adaptation method after loading the untouched Phase-1 checkpoint.
    for p in backbone.parameters(): p.requires_grad=False
    if method=="full":
        for p in backbone.parameters(): p.requires_grad=True
    elif method=="lorma":
        install_lorma(backbone,rank=4,alpha=8.0)
    elif method in {"partial_C", "partial"}:
        if arch=="C":
            for layer in list(backbone.encoder)[-2:]+list(backbone.decoder)[-4:]:
                for p in layer.parameters(): p.requires_grad=True
            for module_name in ("encoder_final_norm","decoder_final_norm","case_memory_attention","case_memory_norm"):
                mod=getattr(backbone,module_name,None)
                if mod is not None:
                    for p in mod.parameters(): p.requires_grad=True
            if hasattr(backbone,"case_memory"): backbone.case_memory.requires_grad=True
            if hasattr(backbone,"decoder_start"): backbone.decoder_start.requires_grad=True
        else:
            groups=_layer_groups(backbone,arch)
            if not groups: raise RuntimeError(f"No supported existing layers for partial fine-tuning on {arch}")
            for _,layers in groups:
                for layer in layers[-4:]:
                    for p in layer.parameters(): p.requires_grad=True
            for nm in ("final_norm","encoder_final_norm","decoder_final_norm"):
                mod=getattr(backbone,nm,None)
                if mod is not None:
                    for p in mod.parameters(): p.requires_grad=True
    elif method in {"progressive_A", "progressive"}:
        groups=_layer_groups(backbone,arch)
        if not groups: raise RuntimeError(f"No supported existing layers for progressive unfreezing on {arch}")
        group_name,layers=groups[-1]
        for layer in layers[-2:]:
            for p in layer.parameters(): p.requires_grad=True
        for nm in ("final_norm","decoder_final_norm"):
            mod=getattr(backbone,nm,None)
            if mod is not None:
                for p in mod.parameters(): p.requires_grad=True
        backbone._cjpe_progressive_group=group_name
    elif method in {"adapters_D", "adapters"}:
        install_adapters(backbone,arch,bottleneck=32,last_n=4)
    else:
        raise ValueError("Unknown adaptation method: "+method)
    backbone=backbone.to(DEVICE)
    print(f"Loaded {arch} from {path} | d_model={cfg.get('d_model')} | context={cfg.get('context_length')} | method={method}")
    del payload,state
    gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return backbone,cfg,str(path)

class CJPEHead(nn.Module):
    def __init__(self,dim):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(2*dim,dim//2),nn.GELU(),nn.Dropout(0.15),nn.Linear(dim//2,2))
    def forward(self,x): return self.net(x)

def backbone_features(backbone,arch,input_ids,attention_mask):
    mask=attention_mask.bool()
    if arch=="C":
        # Use the existing encoder, evidence/case memory, and causal decoder; avoid constructing 40K vocab logits.
        memory=backbone.encode_prompt(src_ids=input_ids,src_attention_mask=mask)
        hidden=backbone.decode_teacher_forced(decoder_input_ids=input_ids,memory=memory,decoder_attention_mask=mask)
    else:
        backbone.lm_head.last_hidden=None
        if arch=="D":
            _=backbone(input_ids=input_ids,prefix_len=input_ids.shape[1],return_logits=True)
        else:
            _=backbone(input_ids=input_ids)
        hidden=backbone.lm_head.last_hidden
        if hidden is None: raise RuntimeError(f"Hidden capture failed for architecture {arch}")
    h=hidden.float()
    m=mask.unsqueeze(-1).float()
    mean=(h*m).sum(dim=1)/m.sum(dim=1).clamp_min(1.0)
    h_for_max=h.masked_fill(~mask.unsqueeze(-1),-1.0e4)
    maxv=h_for_max.max(dim=1).values
    return torch.cat([mean,maxv],dim=-1)

def class_weights(labels):
    cnt=np.bincount(np.asarray(labels,dtype=np.int64),minlength=2).astype(np.float32)
    w=cnt.sum()/(2.0*np.maximum(cnt,1.0))
    return torch.tensor(w,dtype=torch.float32,device=DEVICE)

def evaluate_model(backbone,head,arch,X,M,Y=None,threshold=None,batch_size=EVAL_BATCH_SIZE,prob_temperature=1.0):
    backbone.eval(); head.eval()
    logits_all=[]
    with torch.no_grad():
        for start in range(0,len(X),batch_size):
            ids=X[start:start+batch_size].to(DEVICE,non_blocking=True)
            mask=M[start:start+batch_size].to(DEVICE,non_blocking=True)
            with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=(DEVICE.type=="cuda")):
                feats=backbone_features(backbone,arch,ids,mask)
                logits=head(feats)
            logits_all.append(logits.float().cpu())
    logits=torch.cat(logits_all).numpy()
    probs=torch.softmax(torch.tensor(logits)/max(float(prob_temperature),1e-6),dim=-1).numpy()[:,1]
    if Y is None: return probs
    y=np.asarray(Y,dtype=np.int64)
    thresholds=np.arange(0.05,0.951,0.01) if threshold is None else [float(threshold)]
    candidates=[]
    for t in thresholds:
        pred=(probs>=t).astype(np.int64)
        score=f1_score(y,pred,average="macro",zero_division=0)
        candidates.append((float(score),-abs(float(t)-0.5),float(t),pred))
    score,_,best_t,pred=max(candidates,key=lambda x:(x[0],x[1]))
    metrics={"macro_f1":float(score),"macro_precision":float(precision_score(y,pred,average="macro",zero_division=0)),"macro_recall":float(recall_score(y,pred,average="macro",zero_division=0)),"threshold":float(best_t),"confusion_matrix":confusion_matrix(y,pred,labels=[0,1]).tolist()}
    return metrics,probs

def cpu_state_dict(module):
    return {k:v.detach().cpu().contiguous() for k,v in module.state_dict().items()}

def save_exp_checkpoint(path,backbone,head,meta):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+".tmp")
    torch.save({**meta,"backbone_state_dict":cpu_state_dict(backbone),"head_state_dict":cpu_state_dict(head)},tmp)
    os.replace(tmp,path)

def restore_experiment(ckpt_path):
    meta=torch.load(ckpt_path,map_location="cpu",weights_only=False)
    arch=meta["arch"]; method=meta["method"]
    backbone,cfg,basepath=load_backbone(arch,method,meta["phase1_checkpoint_path"])
    missing,unexpected=backbone.load_state_dict(meta["backbone_state_dict"],strict=False)
    bad_missing=[x for x in missing if not (x.endswith("lm_head.weight") and cfg.get("tie_embeddings",False))]
    if bad_missing or unexpected:
        raise RuntimeError(f"Could not restore experiment checkpoint {ckpt_path}; missing={bad_missing[:8]} unexpected={list(unexpected)[:8]}")
    head=CJPEHead(int(cfg["d_model"])).to(DEVICE)
    head.load_state_dict(meta["head_state_dict"],strict=True)
    return backbone,head,cfg,meta

def write_submission(pred_records,exp_name):
    out_dir=SUB_DIR/exp_name; out_dir.mkdir(parents=True,exist_ok=True)
    prediction_path=out_dir/"prediction.json"
    zip_path=out_dir/"submission.zip"
    obj=[{"id":r["id_value"],"pred":int(p)} for r,p in zip(TEST_RECORDS,pred_records)]
    assert len(obj)==len(TEST_RECORDS)
    assert [str(x["id"]) for x in obj]==[r["id"] for r in TEST_RECORDS]
    assert all(isinstance(x["pred"],int) and x["pred"] in (0,1) for x in obj)
    tmp=prediction_path.with_suffix(".json.tmp")
    tmp.write_text(json.dumps(obj,ensure_ascii=False),encoding="utf-8")
    os.replace(tmp,prediction_path)
    with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(prediction_path,arcname="prediction.json")
    with zipfile.ZipFile(zip_path,"r") as zf:
        assert zf.namelist()==["prediction.json"]
        assert zf.testzip() is None
        reread=json.loads(zf.read("prediction.json").decode("utf-8"))
    assert len(reread)==len(TEST_RECORDS)
    digest=sha256_file(zip_path)
    print("SUBMISSION SAVED:",zip_path,"| examples:",len(obj),"| sha256:",digest)
    return str(prediction_path),str(zip_path),digest

print("CJPE classifier and experiment utilities: READY")

In [ ]:
# Core fixes vs the emergency run:
#   * shuffled epochs cover each row exactly once per epoch (no random-with-replacement batches)
#   * lower learning rate + cosine schedule
#   * uses masked mean, masked max, and last-valid-token pooling in the classification head
#   * checkpoints only trainable adapter weights + head + optimizer/scheduler/scaler/sampler state
#   * writes periodic portable resume bundles and atomically saves the best checkpoint

import traceback
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report

EXPERIMENT_NAME = "E9_A_Adapters_SequentialFullEpoch"
EXP_OUT = EXP_DIR / EXPERIMENT_NAME
EXP_OUT.mkdir(parents=True, exist_ok=True)
RESUME_PATH = EXP_OUT / "resume_state.pt"
BEST_PATH = EXP_OUT / "best_model.pt"
HISTORY_PATH = EXP_OUT / "training_history.csv"
BUNDLE_PATH = ROOT / "CJPE_PHASE4_RESUME_BUNDLE.zip"

TARGET_EPOCHS = 3
SESSION_BUDGET_HOURS = 7.0          # re-run the training cell to continue if this block ends
CHECKPOINT_EVERY_STEPS = 50         # at most 50 updates lost after a clean interruption
BUNDLE_EVERY_STEPS = 250
EVAL_EVERY_STEPS = 500
MIN_EPOCHS_BEFORE_EARLY_STOP = 2
PATIENCE_EVALS = 12
ADAPTER_LR = 1.0e-4
HEAD_LR = 3.0e-4
WEIGHT_DECAY = 0.01
LABEL_SMOOTHING = 0.01

# The backbone architecture is unchanged. This is an improved task-specific pooling/classification head only.
class CJPEHead(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3 * dim, dim // 2),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(dim // 2, 2),
        )
    def forward(self, x):
        return self.net(x)

def backbone_features(backbone, arch, input_ids, attention_mask):
    mask = attention_mask.bool()
    if arch == "C":
        memory = backbone.encode_prompt(src_ids=input_ids, src_attention_mask=mask)
        hidden = backbone.decode_teacher_forced(decoder_input_ids=input_ids, memory=memory, decoder_attention_mask=mask)
    else:
        backbone.lm_head.last_hidden = None
        if arch == "D":
            _ = backbone(input_ids=input_ids, prefix_len=input_ids.shape[1], return_logits=True)
        else:
            _ = backbone(input_ids=input_ids)
        hidden = backbone.lm_head.last_hidden
        if hidden is None: raise RuntimeError(f"Hidden capture failed for architecture {arch}")
    h = hidden.float()
    m = mask.unsqueeze(-1).float()
    mean = (h * m).sum(dim=1) / m.sum(dim=1).clamp_min(1.0)
    maxv = h.masked_fill(~mask.unsqueeze(-1), -1.0e4).max(dim=1).values
    last_index = mask.long().sum(dim=1).clamp_min(1) - 1
    last = h[torch.arange(h.size(0), device=h.device), last_index]
    return torch.cat([mean, maxv, last], dim=-1)

def _atomic_torch_save(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def _trainable_state(module):
    return {name: p.detach().cpu().contiguous().clone() for name, p in module.named_parameters() if p.requires_grad}

def _load_named_parameters(module, state, strict=True):
    params = dict(module.named_parameters())
    missing = []
    with torch.no_grad():
        for name, value in state.items():
            if name not in params:
                if strict: raise RuntimeError(f"Trainable parameter not found on restore: {name}")
                continue
            params[name].copy_(value.to(device=params[name].device, dtype=params[name].dtype))
    if strict:
        missing = [name for name,p in params.items() if p.requires_grad and name not in state]
        if missing: raise RuntimeError(f"Missing trainable parameters on restore: {missing[:10]}")

def _bundle_resume_files():
    # Keep only compact task-specific deltas, not the frozen ~1 GB Phase-1 backbone.
    tmp = BUNDLE_PATH.with_suffix(".zip.tmp")
    with zipfile.ZipFile(tmp, "w", compression=zipfile.ZIP_STORED) as zf:
        for p in (RESUME_PATH, BEST_PATH, HISTORY_PATH):
            if p.exists(): zf.write(p, arcname=p.name)
        manifest = {
            "experiment":EXPERIMENT_NAME,"base_checkpoint":str(CHECKPOINT_PATHS["A"]),
            "data_signature":DATA_SIGNATURE,"tokenizer_sha256":TOKENIZER_SHA256,
            "max_seq_len":MAX_SEQ_LEN,"target_epochs":TARGET_EPOCHS,
        }
        zf.writestr("resume_manifest.json", json.dumps(manifest, indent=2))
    os.replace(tmp, BUNDLE_PATH)

def _restore_bundle_from_attached_input():
    if RESUME_PATH.exists(): return False
    search_root = Path("/kaggle/input")
    if not search_root.exists(): return False
    for p in iter_files_pruned(search_root, names={BUNDLE_PATH.name}):
        try:
            with zipfile.ZipFile(p) as zf:
                safe = {"resume_state.pt", "best_model.pt", "training_history.csv", "resume_manifest.json"}
                for name in zf.namelist():
                    if name in safe:
                        dest = EXP_OUT / name
                        with zf.open(name) as src, open(dest, "wb") as out: shutil.copyfileobj(src, out)
            print("Imported portable resume bundle from", p)
            return RESUME_PATH.exists()
        except Exception as e:
            print("Could not import resume bundle", p, repr(e)[:200])
    return False

def _restore_best_model(path=BEST_PATH):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    if ck.get("data_signature") != DATA_SIGNATURE or ck.get("tokenizer_sha256") != TOKENIZER_SHA256:
        raise RuntimeError("Best checkpoint was trained with a different dataset/tokenizer signature; refusing to use it for this test split.")
    base_path = ck["phase1_checkpoint_path"]
    backbone, cfg, _ = load_backbone("A", "adapters", base_path)
    _load_named_parameters(backbone, ck["trainable_state_dict"], strict=True)
    head = CJPEHead(int(cfg["d_model"])).to(DEVICE)
    head.load_state_dict(ck["head_state_dict"], strict=True)
    return backbone, head, cfg, ck

def _get_dev_metrics(backbone, head):
    backbone.eval(); head.eval(); logits_all = []
    with torch.no_grad():
        for start in range(0, len(DEV_X), EVAL_BATCH_SIZE):
            ids = DEV_X[start:start+EVAL_BATCH_SIZE].to(DEVICE, dtype=torch.long, non_blocking=True)
            mask = DEV_M[start:start+EVAL_BATCH_SIZE].to(DEVICE, dtype=torch.bool, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE.type=="cuda")):
                feats = backbone_features(backbone, "A", ids, mask)
                logits = head(feats)
            logits_all.append(logits.float().cpu())
    logits = torch.cat(logits_all).numpy()
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:,1]
    y = DEV_Y.numpy().astype(np.int64)
    candidates=[]
    for threshold in np.arange(0.05, 0.951, 0.01):
        pred=(probs>=float(threshold)).astype(np.int64)
        mf1=f1_score(y,pred,average="macro",zero_division=0)
        candidates.append((float(mf1),-abs(float(threshold)-0.5),float(threshold),pred))
    score,_,threshold,pred=max(candidates,key=lambda z:(z[0],z[1]))
    metrics={
        "macro_f1":float(score),
        "macro_precision":float(precision_score(y,pred,average="macro",zero_division=0)),
        "macro_recall":float(recall_score(y,pred,average="macro",zero_division=0)),
        "threshold":float(threshold),
        "confusion_matrix":confusion_matrix(y,pred,labels=[0,1]).tolist(),
        "classification_report":classification_report(y,pred,labels=[0,1],output_dict=True,zero_division=0),
    }
    return metrics, probs

def _save_best(backbone, head, step, epoch_completed, metrics, dev_probs, elapsed):
    meta={
        "experiment":EXPERIMENT_NAME,"arch":"A","method":"adapters",
        "phase1_checkpoint_path":str(CHECKPOINT_PATHS["A"]),
        "model_config":CHECKPOINT_META["A"]["cfg"],
        "best_macro_f1":metrics["macro_f1"],"best_threshold":metrics["threshold"],
        "best_dev_metrics":metrics,"best_step":int(step),"completed_epochs":int(epoch_completed),
        "train_count":len(TRAIN_USED),"dev_count":len(DEV_USED),"test_count":len(TEST_RECORDS),
        "trainable_parameters":sum(p.numel() for p in list(backbone.parameters())+list(head.parameters()) if p.requires_grad),
        "data_signature":DATA_SIGNATURE,"tokenizer_sha256":TOKENIZER_SHA256,"max_seq_len":MAX_SEQ_LEN,
        "pooling":"masked mean + masked max + last valid token",
        "context_strategy":"first 256 + last 256 token IDs; attention only causal; pad tokens excluded from pooling",
        "elapsed_minutes":float(elapsed),"timestamp":time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    _atomic_torch_save({**meta,"trainable_state_dict":_trainable_state(backbone),"head_state_dict":{k:v.detach().cpu().clone() for k,v in head.state_dict().items()}}, BEST_PATH)
    np.save(EXP_OUT / "best_dev_positive_probabilities.npy", dev_probs.astype(np.float32))
    (EXP_OUT / "best_dev_metrics.json").write_text(json.dumps(meta,indent=2),encoding="utf-8")
    print("NEW BEST | Macro-F1=",round(metrics["macro_f1"],5),"threshold=",metrics["threshold"],"step=",step,flush=True)

def _save_resume(backbone, head, optimizer, scheduler, scaler, state, force_bundle=False):
    payload={
        "version":2,"experiment":EXPERIMENT_NAME,"arch":"A","method":"adapters",
        "phase1_checkpoint_path":str(CHECKPOINT_PATHS["A"]),"data_signature":DATA_SIGNATURE,
        "tokenizer_sha256":TOKENIZER_SHA256,"max_seq_len":MAX_SEQ_LEN,"batch_size":BATCH_SIZE,
        "target_epochs":TARGET_EPOCHS,"trainable_state_dict":_trainable_state(backbone),
        "head_state_dict":{k:v.detach().cpu().clone() for k,v in head.state_dict().items()},
        "optimizer_state_dict":optimizer.state_dict(),"scheduler_state_dict":scheduler.state_dict(),
        "scaler_state_dict":scaler.state_dict(),"python_rng_state":random.getstate(),
        "numpy_rng_state":np.random.get_state(),"torch_rng_state":torch.get_rng_state(),
        "cuda_rng_state_all":torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
        **state,
    }
    _atomic_torch_save(payload, RESUME_PATH)
    if force_bundle or int(state.get("global_step",0)) % BUNDLE_EVERY_STEPS == 0:
        try: _bundle_resume_files()
        except Exception as e: print("Portable bundle update warning:",repr(e)[:250],flush=True)

def run_resumable_full_epochs():
    _restore_bundle_from_attached_input()
    if DEVICE.type != "cuda":
        print("WARNING: GPU unavailable. Training will be very slow; consider stopping and enabling GPU.",flush=True)
    run_start=time.time(); session_deadline=run_start+SESSION_BUDGET_HOURS*3600
    n=len(TRAIN_X); steps_per_epoch=int(math.ceil(n/BATCH_SIZE)); max_total_steps=steps_per_epoch*TARGET_EPOCHS
    backbone,cfg,basepath=load_backbone("A","adapters",CHECKPOINT_PATHS["A"])
    head=CJPEHead(int(cfg["d_model"])).to(DEVICE)
    # Fail fast on incompatible checkpoint/model code before spending hours on training.
    backbone.eval(); head.eval()
    with torch.no_grad():
        probe_ids=TRAIN_X[:2].to(DEVICE,dtype=torch.long,non_blocking=True)
        probe_mask=TRAIN_M[:2].to(DEVICE,dtype=torch.bool,non_blocking=True)
        with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=(DEVICE.type=="cuda")):
            probe_features=backbone_features(backbone,"A",probe_ids,probe_mask)
            probe_logits=head(probe_features)
        assert probe_logits.shape==(min(2,len(TRAIN_X)),2), f"Unexpected classifier output shape: {tuple(probe_logits.shape)}"
        assert torch.isfinite(probe_logits).all(), "Checkpoint/model smoke test returned non-finite logits"
    print("CHECKPOINT SMOKE TEST PASSED | features=",tuple(probe_features.shape),"logits=",tuple(probe_logits.shape),flush=True)
    del probe_ids,probe_mask,probe_features,probe_logits
    backbone.train(); head.train()
    adapter_params=[p for p in backbone.parameters() if p.requires_grad]
    if not adapter_params: raise RuntimeError("Adapter parameters were not marked trainable")
    optimizer=torch.optim.AdamW([
        {"params":adapter_params,"lr":ADAPTER_LR},
        {"params":list(head.parameters()),"lr":HEAD_LR},
    ], weight_decay=WEIGHT_DECAY, betas=(0.9,0.999), eps=1e-8)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=max_total_steps,eta_min=1.0e-6)
    scaler=torch.cuda.amp.GradScaler(enabled=(DEVICE.type=="cuda"))
    rng=np.random.default_rng(SEED+2026)
    state={"epoch":0,"offset":0,"order":rng.permutation(n).astype(np.int32),"global_step":0,
           "best_f1":-1.0,"best_threshold":0.5,"stale_evals":0,"history":[],"last_loss":None,
           "rng_state":rng.bit_generator.state,"early_stopped":False}
    if RESUME_PATH.exists():
        try:
            ck=torch.load(RESUME_PATH,map_location="cpu",weights_only=False)
            compatible=(ck.get("data_signature")==DATA_SIGNATURE and ck.get("tokenizer_sha256")==TOKENIZER_SHA256
                        and ck.get("max_seq_len")==MAX_SEQ_LEN and ck.get("batch_size")==BATCH_SIZE
                        and ck.get("experiment")==EXPERIMENT_NAME and ck.get("phase1_checkpoint_path")==str(CHECKPOINT_PATHS["A"])
                        and int(ck.get("target_epochs", TARGET_EPOCHS))==TARGET_EPOCHS)
            if compatible:
                _load_named_parameters(backbone,ck["trainable_state_dict"],strict=True)
                head.load_state_dict(ck["head_state_dict"],strict=True)
                optimizer.load_state_dict(ck["optimizer_state_dict"])
                scheduler.load_state_dict(ck["scheduler_state_dict"])
                scaler.load_state_dict(ck.get("scaler_state_dict",{}))
                for k in state:
                    if k in ck: state[k]=ck[k]
                try: random.setstate(ck["python_rng_state"]); np.random.set_state(ck["numpy_rng_state"]); torch.set_rng_state(ck["torch_rng_state"]); rng.bit_generator.state=state["rng_state"]
                except Exception as e: print("RNG restore warning:",repr(e)[:100])
                if torch.cuda.is_available() and ck.get("cuda_rng_state_all"):
                    try: torch.cuda.set_rng_state_all(ck["cuda_rng_state_all"])
                    except Exception: pass
                print(f"RESUMED | epoch={state['epoch']} offset={state['offset']}/{n} step={state['global_step']} best={state['best_f1']:.5f}",flush=True)
            else:
                raise RuntimeError("Existing resume checkpoint does not match this data/config/base checkpoint. Do not overwrite it blindly: restore the matching data/checkpoint inputs or intentionally remove the old E9 resume files to start from scratch.")
        except Exception as e:
            print("Could not restore previous optimizer checkpoint; start new training only after confirming this is expected:",repr(e)[:300],flush=True)
            raise
    else:
        print("STARTING NEW TRAINING RUN | rows per epoch:",n,"| updates/epoch:",steps_per_epoch,"| target epochs:",TARGET_EPOCHS,flush=True)

    class_weight=class_weights(TRAIN_Y.numpy())
    backbone.train(); head.train()
    last_checkpoint_step=int(state["global_step"])
    try:
        while int(state["epoch"]) < TARGET_EPOCHS and not state.get("early_stopped", False):
            if time.time() >= session_deadline:
                print("Session training budget reached; saving and pausing cleanly.",flush=True); break
            off=int(state["offset"]); order=np.asarray(state["order"],dtype=np.int32)
            idx=order[off:min(off+BATCH_SIZE,n)]
            if len(idx)==0:
                state["epoch"]+=1; state["offset"]=0
                if state["epoch"]<TARGET_EPOCHS: state["order"]=rng.permutation(n).astype(np.int32)
                continue
            ids=TRAIN_X[idx].to(DEVICE,dtype=torch.long,non_blocking=True)
            mask=TRAIN_M[idx].to(DEVICE,dtype=torch.bool,non_blocking=True)
            labels=TRAIN_Y[idx].to(DEVICE,dtype=torch.long,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=(DEVICE.type=="cuda")):
                features=backbone_features(backbone,"A",ids,mask)
                logits=head(features)
                loss=F.cross_entropy(logits.float(),labels,weight=class_weight,label_smoothing=LABEL_SMOOTHING)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite training loss at global step {state['global_step']+1}")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(adapter_params+list(head.parameters()),1.0)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            state["global_step"]+=1; state["last_loss"]=float(loss.detach().cpu())
            state["offset"] = off + len(idx)
            epoch_boundary=(int(state["offset"])>=n)
            if epoch_boundary:
                state["epoch"]+=1; state["offset"]=0
                if int(state["epoch"])<TARGET_EPOCHS:
                    state["order"]=rng.permutation(n).astype(np.int32)
                    state["rng_state"]=rng.bit_generator.state
                else:
                    state["order"]=np.arange(n,dtype=np.int32)
            if int(state["global_step"])%25==0:
                ep_display=min(TARGET_EPOCHS, int(state["epoch"])+1 if not epoch_boundary else int(state["epoch"]))
                print(f"{EXPERIMENT_NAME} | epoch={ep_display}/{TARGET_EPOCHS} | step={state['global_step']}/{max_total_steps} | seen={state['offset']:,}/{n:,} current_epoch | loss={state['last_loss']:.5f} | lr={optimizer.param_groups[0]['lr']:.2e}",flush=True)
            do_eval=(int(state["global_step"])%EVAL_EVERY_STEPS==0) or epoch_boundary
            if do_eval:
                metrics,dev_probs=_get_dev_metrics(backbone,head)
                backbone.train(); head.train()
                row={"global_step":int(state["global_step"]),"completed_epochs":int(state["epoch"]),"offset":int(state["offset"]),
                     "train_loss":state["last_loss"],"elapsed_minutes":(time.time()-run_start)/60.0,**{k:v for k,v in metrics.items() if k!="classification_report"}}
                state["history"].append(row)
                pd.DataFrame(state["history"]).to_csv(HISTORY_PATH,index=False)
                print(f"DEV | epoch={state['epoch']} step={state['global_step']} Macro-F1={metrics['macro_f1']:.5f} P={metrics['macro_precision']:.5f} R={metrics['macro_recall']:.5f} threshold={metrics['threshold']:.2f} CM={metrics['confusion_matrix']}",flush=True)
                if metrics["macro_f1"] > float(state["best_f1"])+1e-6:
                    state["best_f1"]=float(metrics["macro_f1"]); state["best_threshold"]=float(metrics["threshold"]); state["stale_evals"]=0
                    _save_best(backbone,head,state["global_step"],state["epoch"],metrics,dev_probs,(time.time()-run_start)/60)
                else:
                    state["stale_evals"]+=1
                _save_resume(backbone,head,optimizer,scheduler,scaler,state,force_bundle=True)
                if int(state["epoch"])>=MIN_EPOCHS_BEFORE_EARLY_STOP and int(state["stale_evals"])>=PATIENCE_EVALS:
                    state["early_stopped"] = True
                    print("Early stopping after minimum epoch requirement and validation patience.",flush=True); break
            if int(state["global_step"])%CHECKPOINT_EVERY_STEPS==0:
                _save_resume(backbone,head,optimizer,scheduler,scaler,state,force_bundle=False)
            if int(state["global_step"])%BUNDLE_EVERY_STEPS==0:
                try: _bundle_resume_files()
                except Exception as e: print("Bundle save warning:",repr(e)[:200],flush=True)
        # Always save current state even when we pause due to session budget.
        _save_resume(backbone,head,optimizer,scheduler,scaler,state,force_bundle=True)
        pd.DataFrame(state["history"]).to_csv(HISTORY_PATH,index=False)
    except Exception:
        print("TRAINING ERROR — saving last safe state before propagating the error:",flush=True)
        try: _save_resume(backbone,head,optimizer,scheduler,scaler,state,force_bundle=True)
        except Exception as save_e: print("Checkpoint save also failed:",repr(save_e)[:300],flush=True)
        traceback.print_exc()
        raise
    finally:
        del optimizer,scheduler,scaler,backbone,head
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    # Generate a validated provisional/final submission from the best completed dev checkpoint.
    if BEST_PATH.exists():
        best_backbone,best_head,best_cfg,best_ck=_restore_best_model()
        best_backbone.eval(); best_head.eval(); test_logits=[]
        with torch.no_grad():
            for start in range(0,len(TEST_X),EVAL_BATCH_SIZE):
                ids=TEST_X[start:start+EVAL_BATCH_SIZE].to(DEVICE,dtype=torch.long,non_blocking=True)
                mask=TEST_M[start:start+EVAL_BATCH_SIZE].to(DEVICE,dtype=torch.bool,non_blocking=True)
                with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=(DEVICE.type=="cuda")):
                    feats=backbone_features(best_backbone,"A",ids,mask)
                    logits=best_head(feats)
                test_logits.append(logits.float().cpu())
        test_probs=torch.softmax(torch.cat(test_logits),dim=-1).numpy()[:,1]
        test_pred=(test_probs>=float(best_ck["best_threshold"])).astype(np.int64)
        pred_path,zip_path,digest=write_submission(test_pred.tolist(),EXPERIMENT_NAME)
        # Promote only after the zip contract has passed within write_submission.
        temp=SUB_DIR/"submission.zip.tmp"
        shutil.copy2(zip_path,temp); os.replace(temp,SUB_DIR/"submission.zip")
        np.save(EXP_OUT/"test_positive_probabilities.npy",test_probs.astype(np.float32))
        print("PROMOTED SUBMISSION:",SUB_DIR/"submission.zip","| SHA256:",digest,flush=True)
        del best_backbone,best_head; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    else:
        print("No validation-selected best checkpoint yet. Resume training; a submission will be emitted after the first full-dev evaluation.",flush=True)
    try: _bundle_resume_files()
    except Exception as e: print("Final bundle update warning:",repr(e)[:200],flush=True)
    print("TRAINING BLOCK FINISHED | completed epochs:",state["epoch"],"| global step:",state["global_step"],"| best dev F1:",state["best_f1"],"| resume:",RESUME_PATH,"| bundle:",BUNDLE_PATH,flush=True)


In [ ]:
# Safe to re-run this cell. It restores the optimizer, scheduler, AMP scaler, adapter weights, head, epoch order and cursor.
# Target: three complete passes over all 37,387 training rows, unless early stopping triggers after at least two epochs.
run_resumable_full_epochs()


In [ ]:
from IPython.display import display, FileLink
summary_paths = [
    SUB_DIR / "submission.zip",
    BUNDLE_PATH,
    BEST_PATH,
    RESUME_PATH,
    EXP_OUT / "best_dev_metrics.json",
    HISTORY_PATH,
    ROOT / "data_manifest.json",
]
for p in summary_paths:
    if p.exists():
        print(f"AVAILABLE | {p} | {p.stat().st_size/1024/1024:.2f} MB")
        display(FileLink(str(p)))
    else:
        print("NOT CREATED YET:", p)
if (EXP_OUT / "best_dev_metrics.json").exists():
    print("\nBEST DEVELOPMENT METRICS:")
    print((EXP_OUT / "best_dev_metrics.json").read_text(encoding="utf-8"))
print("\nTo resume in this same working directory: rerun CELL 7. To resume in a fresh Kaggle session, download CJPE_PHASE4_RESUME_BUNDLE.zip, attach it as an input, then Run All; tokenization is cached separately and can be rebuilt if needed.")


In [ ]:
# The current focused run only trains Architecture A. If compatible B/C/D experiment results are later produced
# under the same data signature, this cell tests their equal-weight probability ensembles. It never ensembles
# arrays with a different dev/test order and never promotes an ensemble that does not beat the strongest member.
from itertools import combinations

def run_dev_selected_ensemble():
    candidates=[]
    for meta_path in EXP_DIR.glob("*/best_dev_metrics.json"):
        try:
            meta=json.loads(meta_path.read_text(encoding="utf-8"))
            if meta.get("data_signature") != DATA_SIGNATURE: continue
            exp=meta.get("experiment", meta_path.parent.name)
            dp=meta_path.parent/"best_dev_positive_probabilities.npy"
            tp=meta_path.parent/"test_positive_probabilities.npy"
            if not dp.exists() or not tp.exists(): continue
            devp=np.load(dp).astype(np.float32); testp=np.load(tp).astype(np.float32)
            if len(devp)!=len(DEV_USED) or len(testp)!=len(TEST_RECORDS): continue
            if not np.isfinite(devp).all() or not np.isfinite(testp).all(): continue
            y=DEV_Y.numpy().astype(np.int64); best=(0.0,0.5)
            for t in np.arange(0.05,0.951,0.01):
                score=f1_score(y,(devp>=t).astype(np.int64),average="macro",zero_division=0)
                if score>best[0]+1e-12 or (abs(score-best[0])<1e-12 and abs(t-0.5)<abs(best[1]-0.5)): best=(float(score),float(t))
            candidates.append({"experiment":exp,"dev":devp,"test":testp,"f1":best[0],"threshold":best[1]})
        except Exception as e:
            print("Skipping ensemble candidate",meta_path,repr(e)[:160])
    if not candidates:
        print("No metadata-compatible model probability files found. Keep the best single-model submission.")
        return
    candidates.sort(key=lambda c:c["f1"],reverse=True)
    print("Compatible candidates:",[(c["experiment"],round(c["f1"],5),c["threshold"]) for c in candidates])
    y=DEV_Y.numpy().astype(np.int64)
    best_single=candidates[0]
    best_ens={"f1":best_single["f1"],"threshold":best_single["threshold"],"dev":best_single["dev"],"test":best_single["test"],"members":[best_single["experiment"]]}
    top=candidates[:min(5,len(candidates))]
    for k in range(2,min(4,len(top))+1):
        for subset in combinations(top,k):
            devp=np.mean(np.stack([c["dev"] for c in subset]),axis=0)
            testp=np.mean(np.stack([c["test"] for c in subset]),axis=0)
            for t in np.arange(0.05,0.951,0.01):
                score=float(f1_score(y,(devp>=t).astype(np.int64),average="macro",zero_division=0))
                if score>best_ens["f1"]+1e-6 or (abs(score-best_ens["f1"])<=1e-6 and len(subset)>len(best_ens["members"]) and abs(t-0.5)<abs(best_ens["threshold"]-0.5)):
                    best_ens={"f1":score,"threshold":float(t),"dev":devp,"test":testp,"members":[c["experiment"] for c in subset]}
    report={"selected_members":best_ens["members"],"dev_macro_f1":best_ens["f1"],"threshold":best_ens["threshold"],
            "selection":"equal-weight probability mean; threshold selected on full provided development split",
            "candidate_scores":[{"experiment":c["experiment"],"dev_macro_f1":c["f1"],"threshold":c["threshold"]} for c in candidates]}
    (ROOT/"ensemble_selection.json").write_text(json.dumps(report,indent=2),encoding="utf-8")
    print("Selected development candidate:",json.dumps(report,indent=2))
    # Only replace the default archive when there are at least two models and the ensemble beats best single by >=0.002.
    if len(best_ens["members"])>=2 and best_ens["f1"] >= best_single["f1"]+0.002:
        preds=(best_ens["test"]>=best_ens["threshold"]).astype(np.int64)
        pred_path,zip_path,digest=write_submission(preds.tolist(),"E10_DevSelectedEnsemble")
        tmp=SUB_DIR/"submission.zip.tmp"; shutil.copy2(zip_path,tmp); os.replace(tmp,SUB_DIR/"submission.zip")
        np.save(EXP_DIR/"E10_DevSelectedEnsemble"/"best_dev_positive_probabilities.npy",best_ens["dev"].astype(np.float32))
        np.save(EXP_DIR/"E10_DevSelectedEnsemble"/"test_positive_probabilities.npy",best_ens["test"].astype(np.float32))
        print("ENSEMBLE PROMOTED | dev Macro-F1=",round(best_ens["f1"],5),"| SHA256=",digest,flush=True)
    else:
        print("No ensemble beats the best single model by at least 0.002; leave the default submission unchanged.",flush=True)

run_dev_selected_ensemble()
